In [1]:
# ============================================================
# FASE 0 - BLOCCO 1: Setup ambiente (percorso corretto)
# ============================================================
import pandas as pd
import duckdb
from pathlib import Path

# --- Percorso base del progetto di gruppo ---
BASE_DIR = Path(r"C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism")

RAW_DIR = BASE_DIR / "data" / "raw"
STAGING_DIR = BASE_DIR / "data" / "staging"
DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"

for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Struttura cartelle creata dentro il progetto di gruppo:")
for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    print(f"  {d.resolve()}")

Struttura cartelle creata dentro il progetto di gruppo:
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\staging
  C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\db


In [2]:
# ============================================================
# FASE 0 - BLOCCO 2: Connessione DB e schema
# ============================================================
con = duckdb.connect(str(DB_PATH))

con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE SCHEMA IF NOT EXISTS staging")
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

# Verifica
schemi = con.execute("""
    SELECT schema_name 
    FROM information_schema.schemata 
    WHERE schema_name IN ('raw','staging','presentation')
""").df()
print(schemi)

    schema_name
0  presentation
1           raw
2       staging


In [3]:
# ============================================================
# FASE 0 - BLOCCO 3a: Download anagrafica comuni ISTAT (scoperta struttura)
# ============================================================
ISTAT_COMUNI_URL = "https://www.istat.it/storage/codici-unita-amministrative/Elenco-comuni-italiani.csv"

def scarica_anagrafica_comuni(url=ISTAT_COMUNI_URL):
    try:
        df = pd.read_csv(url, sep=';', encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(url, sep=';', encoding='cp1252')
    return df

df_comuni_italia = scarica_anagrafica_comuni()

print(f"Righe totali (comuni italiani): {len(df_comuni_italia)}")
print(f"\nColonne disponibili:")
for c in df_comuni_italia.columns:
    print(f"  - {c}")
print(f"\nPrime righe:")
df_comuni_italia.head(3)

Righe totali (comuni italiani): 7896

Colonne disponibili:
  - Codice Regione
  - Codice dell'Unità territoriale sovracomunale 
(valida a fini statistici)
  - Codice Provincia (Storico)(1)
  - Progressivo del Comune (2)
  - Codice Comune formato alfanumerico
  - Denominazione (Italiana e straniera)
  - Denominazione in italiano
  - Denominazione altra lingua
  - Codice Ripartizione Geografica
  - Ripartizione geografica
  - Denominazione Regione
  - Denominazione dell'Unità territoriale sovracomunale 
(valida a fini statistici)
  - Tipologia di Unità territoriale sovracomunale 
  - Flag Comune capoluogo di provincia/città metropolitana/libero consorzio
  - Sigla automobilistica
  - Codice Comune formato numerico
  - Codice Comune numerico con 110 province (dal 2010 al 2016)
  - Codice Comune numerico con 107 province (dal 2006 al 2009)
  - Codice Comune numerico con 103 province (dal 1995 al 2005)
  - Codice Catastale del comune
  - Codice NUTS1 2021
  - Codice NUTS2 2021 (3) 
  - Codi

,Codice Regione,Codice dell'Unità territoriale sovracomunale \n(valida a fini statistici),Codice Provincia (Storico)(1),Progressivo del Comune (2),Codice Comune formato alfanumerico,Denominazione (Italiana e straniera),Denominazione in italiano,Denominazione altra lingua,Codice Ripartizione Geografica,Ripartizione geografica,...,Codice Comune numerico con 110 province (dal 2010 al 2016),Codice Comune numerico con 107 province (dal 2006 al 2009),Codice Comune numerico con 103 province (dal 1995 al 2005),Codice Catastale del comune,Codice NUTS1 2021,Codice NUTS2 2021 (3),Codice NUTS3 2021,Codice NUTS1 2024,Codice NUTS2 2024 (3),Codice NUTS3 2024
0,1,201,1,1,1001,Agliè,Agliè,NaN,1,Nord-ovest,...,1001,1001,1001,A074,ITC,ITC1,ITC11,ITC,ITC1,ITC11
1,1,201,1,2,1002,Airasca,Airasca,NaN,1,Nord-ovest,...,1002,1002,1002,A109,ITC,ITC1,ITC11,ITC,ITC1,ITC11
2,1,201,1,3,1003,Ala di Stura,Ala di Stura,NaN,1,Nord-ovest,...,1003,1003,1003,A117,ITC,ITC1,ITC11,ITC,ITC1,ITC11


In [4]:
# ============================================================
# FASE 0 - BLOCCO 3b: Filtro Sardegna + tabella raw
# ============================================================

# Filtro sulla Denominazione Regione (più leggibile e robusto 
# rispetto a ricordare a memoria il codice numerico regione)
df_comuni_sardegna = df_comuni_italia[
    df_comuni_italia["Denominazione Regione"].str.strip() == "Sardegna"
].copy()

print(f"Comuni Sardegna trovati: {len(df_comuni_sardegna)}")

# Selezioniamo solo le colonne che ci servono davvero per il progetto,
# rinominandole in modo pulito per l'uso successivo (snake_case, italiano semplice)
colonne_utili = {
    "Codice Comune formato alfanumerico": "codice_istat_comune",
    "Codice Comune formato numerico": "codice_istat_numerico",
    "Denominazione in italiano": "comune",
    "Sigla automobilistica": "provincia_sigla",
    "Codice Provincia (Storico)(1)": "codice_provincia",
    "Ripartizione geografica": "ripartizione_geografica",
}

df_anagrafica = df_comuni_sardegna[list(colonne_utili.keys())].rename(columns=colonne_utili)

# Controllo qualità: nessun codice ISTAT duplicato o nullo (è la nostra chiave di join futura)
n_duplicati = df_anagrafica["codice_istat_comune"].duplicated().sum()
n_nulli = df_anagrafica["codice_istat_comune"].isna().sum()
print(f"Codici ISTAT duplicati: {n_duplicati}")
print(f"Codici ISTAT nulli: {n_nulli}")

df_anagrafica.head(10)


Comuni Sardegna trovati: 377
Codici ISTAT duplicati: 0
Codici ISTAT nulli: 0


,codice_istat_comune,codice_istat_numerico,comune,provincia_sigla,codice_provincia,ripartizione_geografica
7519,90001,90001,Aggius,SS,90,Isole
7520,90002,90002,Alà dei Sardi,SS,90,Isole
7521,90003,90003,Alghero,SS,90,Isole
7522,90004,90004,Anela,SS,90,Isole
7523,90005,90005,Ardara,SS,90,Isole
7524,90006,90006,Arzachena,SS,90,Isole
7525,90007,90007,Banari,SS,90,Isole
7526,90008,90008,Benetutti,SS,90,Isole
7527,90009,90009,Berchidda,SS,90,Isole
7528,90010,90010,Bessude,SS,90,Isole


In [5]:
# ============================================================
# FASE 0 - BLOCCO 3c: Salvataggio in DuckDB (schema raw)
# ============================================================
con.execute("DROP TABLE IF EXISTS raw.anagrafica_comuni")
con.execute("CREATE TABLE raw.anagrafica_comuni AS SELECT * FROM df_anagrafica")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, 
           COUNT(DISTINCT codice_istat_comune) AS n_codici_unici
    FROM raw.anagrafica_comuni
""").df()
print(verifica)

con.execute("SELECT * FROM raw.anagrafica_comuni LIMIT 5").df()

   n_comuni  n_codici_unici
0       377             377


,codice_istat_comune,codice_istat_numerico,comune,provincia_sigla,codice_provincia,ripartizione_geografica
0,90001,90001,Aggius,SS,90,Isole
1,90002,90002,Alà dei Sardi,SS,90,Isole
2,90003,90003,Alghero,SS,90,Isole
3,90004,90004,Anela,SS,90,Isole
4,90005,90005,Ardara,SS,90,Isole


In [6]:
# ============================================================
# RE-RUN FASE 0 - Blocco 1: Setup ambiente
# ============================================================
import pandas as pd
import duckdb
from pathlib import Path

BASE_DIR = Path(r"C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism")
RAW_DIR = BASE_DIR / "data" / "raw"
STAGING_DIR = BASE_DIR / "data" / "staging"
DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"

for d in [RAW_DIR, STAGING_DIR, DB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Cartelle pronte.")

Cartelle pronte.


In [7]:
# ============================================================
# RE-RUN FASE 0 - Blocco 2: Connessione DuckDB
# ============================================================
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE SCHEMA IF NOT EXISTS staging")
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")
print("Connessione DB attiva.")

Connessione DB attiva.


In [8]:
# ============================================================
# FASE 1 - BLOCCO B: Porti + Aeroporti (caricamento e pulizia)
# ============================================================
import pandas as pd

# Percorso del file così come l'hai salvato nel repo
PATH_RAW_PORTI_AEROPORTI = RAW_DIR / "bollettino_arrivi_partenze(1).csv"

df_porti_aeroporti = pd.read_csv(PATH_RAW_PORTI_AEROPORTI, sep=',', encoding='utf-8')

# Conversione data (formato dd/mm/yyyy)
df_porti_aeroporti["data"] = pd.to_datetime(df_porti_aeroporti["data"], format="%d/%m/%Y")

print(f"Righe totali nel file originale: {len(df_porti_aeroporti)}")
print(f"Periodo coperto: {df_porti_aeroporti['data'].min().date()} — {df_porti_aeroporti['data'].max().date()}")
print(f"Scali presenti: {sorted(df_porti_aeroporti['nome'].unique())}")

# Filtro sugli anni richiesti: 2022-2026
df_filtrato = df_porti_aeroporti[df_porti_aeroporti["data"].dt.year >= 2022].copy()

print(f"\nRighe dopo filtro 2022-2026: {len(df_filtrato)}")
print(f"Periodo effettivo dopo filtro: {df_filtrato['data'].min().date()} — {df_filtrato['data'].max().date()}")

df_filtrato.head(10)

Righe totali nel file originale: 19865
Periodo coperto: 2019-01-01 — 2026-07-11
Scali presenti: ['Alghero', 'Arbatax', 'Cagliari', 'Golfo Aranci', 'Olbia', 'Porto Torres', 'Porto Vesme']

Righe dopo filtro 2022-2026: 12132
Periodo effettivo dopo filtro: 2022-01-01 — 2026-07-11


,data,porto/aeroporto,nome,arrivi,partenze
0,2026-07-11,Aeroporto,Olbia,14279,12967
1,2026-07-11,Aeroporto,Cagliari,12825,11796
2,2026-07-11,Aeroporto,Alghero,4614,4004
3,2026-07-10,Aeroporto,Olbia,16723,12357
4,2026-07-10,Aeroporto,Cagliari,12375,10942
5,2026-07-10,Aeroporto,Alghero,5187,4519
6,2026-07-09,Porto,Porto Torres,1565,1200
7,2026-07-09,Porto,Olbia,8985,7117
8,2026-07-09,Porto,Golfo Aranci,1082,1042
9,2026-07-09,Porto,Cagliari,664,764


In [9]:
# ============================================================
# FASE 1 - BLOCCO C: Controllo qualità dati
# ============================================================

# Verifica valori nulli
print("Valori nulli per colonna:")
print(df_filtrato.isnull().sum())

# Verifica righe duplicate (stessa data+scalo due volte sarebbe un problema)
duplicati = df_filtrato.duplicated(subset=["data", "porto/aeroporto", "nome"]).sum()
print(f"\nRighe duplicate (data+scalo): {duplicati}")

# Controllo completezza: quanti giorni distinti per ciascuno scalo
completezza = df_filtrato.groupby(["porto/aeroporto", "nome"])["data"].agg(
    n_giorni="count",
    prima_data="min",
    ultima_data="max"
).reset_index()
print("\nCompletezza per scalo:")
completezza

Valori nulli per colonna:
data               0
porto/aeroporto    0
nome               0
arrivi             0
partenze           0
dtype: int64

Righe duplicate (data+scalo): 0

Completezza per scalo:


,porto/aeroporto,nome,n_giorni,prima_data,ultima_data
0,Aeroporto,Alghero,1644,2022-01-01,2026-07-11
1,Aeroporto,Cagliari,1644,2022-01-01,2026-07-11
2,Aeroporto,Olbia,1635,2022-01-01,2026-07-11
3,Porto,Arbatax,970,2022-01-02,2026-07-09
4,Porto,Cagliari,1607,2022-01-01,2026-07-09
5,Porto,Golfo Aranci,1249,2022-01-01,2026-07-09
6,Porto,Olbia,1595,2022-01-01,2026-07-09
7,Porto,Porto Torres,1566,2022-01-01,2026-07-09
8,Porto,Porto Vesme,222,2025-01-01,2026-07-06


In [10]:
# ============================================================
# FASE 1 - BLOCCO C-bis: Dettaglio giorni mancanti per scalo
# ============================================================
import pandas as pd

def giorni_mancanti_per_scalo(df, tipo, nome_scalo, data_inizio=None, data_fine=None):
    subset = df[(df["porto/aeroporto"] == tipo) & (df["nome"] == nome_scalo)]
    
    d_inizio = data_inizio or subset["data"].min()
    d_fine = data_fine or subset["data"].max()
    
    tutte_le_date = pd.date_range(start=d_inizio, end=d_fine, freq="D")
    date_presenti = set(subset["data"])
    date_mancanti = sorted(set(tutte_le_date) - date_presenti)
    
    print(f"\n--- {tipo} {nome_scalo} ---")
    print(f"Periodo atteso: {d_inizio.date()} — {d_fine.date()} ({len(tutte_le_date)} giorni)")
    print(f"Giorni mancanti: {len(date_mancanti)}")
    
    if date_mancanti:
        df_mancanti = pd.DataFrame({"data_mancante": date_mancanti})
        df_mancanti["anno"] = df_mancanti["data_mancante"].dt.year
        df_mancanti["mese"] = df_mancanti["data_mancante"].dt.month
        
        riepilogo = df_mancanti.groupby(["anno", "mese"]).size().reset_index(name="n_giorni_mancanti")
        print("Distribuzione mancanze per anno/mese:")
        print(riepilogo.to_string(index=False))
    
    return date_mancanti

# Golfo Aranci - il caso più sospetto
mancanti_golfo_aranci = giorni_mancanti_per_scalo(df_filtrato, "Porto", "Golfo Aranci")

# Aeroporto Olbia - scarto minore, ma controlliamo comunque
mancanti_olbia_aeroporto = giorni_mancanti_per_scalo(df_filtrato, "Aeroporto", "Olbia")


--- Porto Golfo Aranci ---
Periodo atteso: 2022-01-01 — 2026-07-09 (1651 giorni)
Giorni mancanti: 402
Distribuzione mancanze per anno/mese:
 anno  mese  n_giorni_mancanti
 2022     1                  3
 2022     2                 17
 2022     3                 20
 2022     4                  1
 2022     5                  2
 2022     9                  4
 2022    10                  1
 2022    11                 16
 2022    12                 12
 2023     1                 13
 2023     2                 16
 2023     3                 22
 2023     4                  2
 2023     8                  3
 2023     9                  2
 2023    11                 16
 2023    12                 16
 2024     1                 18
 2024     2                 19
 2024     3                 17
 2024     4                  8
 2024     5                  3
 2024     9                  1
 2024    10                 11
 2024    11                 17
 2024    12                 10
 2025     1           

In [11]:
# ============================================================
# FASE 1 - BLOCCO D: Salvataggio pulito e carico in DuckDB (raw)
# ============================================================

# Salvataggio versione filtrata/pulita in RAW_DIR (nome semplice, come da convenzione scelta)
PATH_OUTPUT = RAW_DIR / "porti_aeroporti.csv"
df_filtrato.to_csv(PATH_OUTPUT, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT}")

# Carico in DuckDB, schema raw
con.execute("DROP TABLE IF EXISTS raw.porti_aeroporti")
con.execute(f"""
    CREATE TABLE raw.porti_aeroporti AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           MIN(data) AS data_min,
           MAX(data) AS data_max,
           COUNT(DISTINCT nome) AS n_scali
    FROM raw.porti_aeroporti
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\porti_aeroporti.csv
   n_righe   data_min   data_max  n_scali
0    12132 2022-01-01 2026-07-11        7


In [12]:
# ============================================================
# FASE 1 - BLOCCO B: Arrivi/Presenze SIRED - ispezione 2025
# ============================================================
import pandas as pd

PATH_SIRED_2025 = RAW_DIR / "csv_opendata_comuni_2025.csv"

df_sired_2025 = pd.read_csv(PATH_SIRED_2025, sep=',', encoding='utf-8')

print(f"Righe totali: {len(df_sired_2025)}")
print(f"Comuni distinti: {df_sired_2025['comune'].nunique()}")
print(f"Mesi presenti: {sorted(df_sired_2025['mese'].unique(), key=str)}")
print(f"\nMacro-tipologie:")
print(df_sired_2025['macro-tipologia'].value_counts())

print(f"\nRighe con 'non disponibile' in mese o macro-tipologia:")
mask_non_disp = (df_sired_2025['mese'] == 'non disponibile') | (df_sired_2025['macro-tipologia'] == 'non disponibile')
print(f"  Totale righe: {mask_non_disp.sum()} ({mask_non_disp.sum()/len(df_sired_2025)*100:.2f}%)")
print(f"  Comuni coinvolti: {df_sired_2025[mask_non_disp]['comune'].nunique()}")
print(f"  Esempi di comuni coinvolti: {df_sired_2025[mask_non_disp]['comune'].unique()[:15]}")

print(f"\nProvenienze distinte: {df_sired_2025['provenienza'].nunique()}")

Righe totali: 125422
Comuni distinti: 323
Mesi presenti: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9', 'non disponibile']

Macro-tipologie:
macro-tipologia
Esercizi Extra-Alberghieri:Alloggi privati in affitto    44883
Esercizi Alberghieri                                     40149
Esercizi Extra-Alberghieri:Esercizi Complementari        39050
non disponibile                                           1340
Name: count, dtype: int64

Righe con 'non disponibile' in mese o macro-tipologia:
  Totale righe: 1340 (1.07%)
  Comuni coinvolti: 256
  Esempi di comuni coinvolti: <ArrowStringArray>
[          'Anela',       'Bonnanaro',         'Bonorva',     'Bortigiadas',
     'Calangianus',       'Cheremule',    'Codrongianos',           'Luras',
 'non disponibile',          'Martis',         'Oschiri',            'Ossi',
          'Padria',        'Perfugas',         'Ploaghe']
Length: 15, dtype: str

Provenienze distinte: 80


In [13]:
# ============================================================
# FASE 1 - BLOCCO C: Verifica righe con comune non attribuibile
# ============================================================

righe_comune_non_disp = df_sired_2025[df_sired_2025['comune'] == 'non disponibile']
print(f"Righe con comune = 'non disponibile': {len(righe_comune_non_disp)}")
print(f"Arrivi totali coinvolti: {righe_comune_non_disp['arrivi'].sum()}")
print(f"Presenze totali coinvolte: {righe_comune_non_disp['presenze'].sum()}")
righe_comune_non_disp.head(10)

Righe con comune = 'non disponibile': 11
Arrivi totali coinvolti: 24
Presenze totali coinvolte: 280


,anno,provincia,comune,mese,macro-tipologia,provenienza,arrivi,presenze
379,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Marche,1,3
8456,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Stati Uniti d'America,3,3
29264,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Francia,3,3
63133,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,1,6
76566,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Repubblica Ceca,1,3
80384,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,0,10
92130,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Toscana,4,8
103908,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,2,2
107918,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Romania,8,240
115954,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,0,0


In [14]:
# ============================================================
# FASE 1 - BLOCCO C: Verifica righe con comune non attribuibile
# ============================================================

righe_comune_non_disp = df_sired_2025[df_sired_2025['comune'] == 'non disponibile']
print(f"Righe con comune = 'non disponibile': {len(righe_comune_non_disp)}")
print(f"Arrivi totali coinvolti: {righe_comune_non_disp['arrivi'].sum()}")
print(f"Presenze totali coinvolte: {righe_comune_non_disp['presenze'].sum()}")
righe_comune_non_disp.head(10)

Righe con comune = 'non disponibile': 11
Arrivi totali coinvolti: 24
Presenze totali coinvolte: 280


,anno,provincia,comune,mese,macro-tipologia,provenienza,arrivi,presenze
379,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Marche,1,3
8456,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Stati Uniti d'America,3,3
29264,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Francia,3,3
63133,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,1,6
76566,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Repubblica Ceca,1,3
80384,2025,Provincia di Oristano,non disponibile,non disponibile,non disponibile,Francia,0,10
92130,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Toscana,4,8
103908,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,2,2
107918,2025,Provincia di Nuoro,non disponibile,non disponibile,non disponibile,Romania,8,240
115954,2025,Città Metropolitana di Sassari,non disponibile,non disponibile,non disponibile,Sardegna,0,0


In [15]:
# ============================================================
# FASE 1 - BLOCCO D: Caricamento raw.arrivi_presenze_sired
# ============================================================

con.execute("DROP TABLE IF EXISTS raw.arrivi_presenze_sired")
con.execute(f"""
    CREATE TABLE raw.arrivi_presenze_sired AS 
    SELECT * FROM read_csv_auto('{PATH_SIRED_2025.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi) AS arrivi_totali,
           SUM(presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired
""").df()
print(verifica)

   n_righe  n_comuni  n_anni  arrivi_totali  presenze_totali
0   125422       323       1      5170093.0       21922765.0


In [16]:
# ============================================================
# FASE 1 - BLOCCO E: Armonizzazione e concatenazione Arrivi/Presenze
# ============================================================
import pandas as pd

file_per_anno = {
    2022: RAW_DIR / "csv_opendata_comuni_2022.csv",
    2023: RAW_DIR / "csv_opendata_comuni_2023.csv",
    2024: RAW_DIR / "csv_opendata_comuni_2024.csv",
    2025: RAW_DIR / "csv_opendata_comuni_2025.csv",
}

def carica_e_armonizza(path, anno):
    df = pd.read_csv(path, sep=',', encoding='utf-8')
    
    # Armonizza nomi colonna (il 2022 usa macro_tipologia con underscore)
    df.columns = [c.strip().lower().replace('_', '-') if c.strip().lower() == 'macro_tipologia' else c.strip() for c in df.columns]
    df = df.rename(columns={"macro_tipologia": "macro-tipologia"})
    
    # Armonizza casing del nome comune (Title Case ovunque)
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    
    # Armonizza casing della macro-tipologia "non disponibile"
    df["macro-tipologia"] = df["macro-tipologia"].astype(str).str.strip()
    df.loc[df["macro-tipologia"].str.lower() == "non disponibile", "macro-tipologia"] = "non disponibile"
    
    # Armonizza colonna mese (assicura sia stringa, per coerenza con 'non disponibile')
    df["mese"] = df["mese"].astype(str).str.strip()
    
    df["anno_verifica"] = anno  # controllo incrociato con la colonna anno originale
    return df

liste_df = []
for anno, path in file_per_anno.items():
    df_anno = carica_e_armonizza(path, anno)
    print(f"{anno}: {len(df_anno)} righe, colonne: {df_anno.columns.tolist()}")
    liste_df.append(df_anno)

df_sired_completo = pd.concat(liste_df, ignore_index=True)

# Controllo coerenza: la colonna 'anno' originale deve combaciare con l'anno del file
disallineati = df_sired_completo[df_sired_completo["anno"].astype(str) != df_sired_completo["anno_verifica"].astype(str)]
print(f"\nRighe con anno disallineato: {len(disallineati)}")

df_sired_completo = df_sired_completo.drop(columns=["anno_verifica"])
print(f"\nTotale righe concatenate: {len(df_sired_completo)}")
print(f"Comuni distinti: {df_sired_completo['comune'].nunique()}")
print(f"Anni coperti: {sorted(df_sired_completo['anno'].unique())}")
df_sired_completo.groupby("anno")[["arrivi", "presenze"]].sum()

2022: 75448 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2023: 98370 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2024: 110498 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']
2025: 125422 righe, colonne: ['anno', 'provincia', 'comune', 'mese', 'macro-tipologia', 'provenienza', 'arrivi', 'presenze', 'anno_verifica']

Righe con anno disallineato: 0

Totale righe concatenate: 409738
Comuni distinti: 331
Anni coperti: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,arrivi,presenze
anno,,
2022,3720188.0,16387935
2023,3904389.0,16337893
2024,4442111.0,18907165
2025,5170093.0,21922765


In [17]:
# ============================================================
# FASE 1 - BLOCCO E-bis: Salvataggio Arrivi/Presenze in raw
# ============================================================

# Escludiamo le righe con comune = 'non disponibile' (decisione presa insieme prima)
df_sired_pulito = df_sired_completo[df_sired_completo["comune"] != "Non Disponibile"].copy()
print(f"Righe escluse (comune non disponibile): {len(df_sired_completo) - len(df_sired_pulito)}")

PATH_OUTPUT_SIRED = RAW_DIR / "arrivi_presenze_sired.csv"
df_sired_pulito.to_csv(PATH_OUTPUT_SIRED, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_SIRED}")

con.execute("DROP TABLE IF EXISTS raw.arrivi_presenze_sired")
con.execute(f"""
    CREATE TABLE raw.arrivi_presenze_sired AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_SIRED.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi) AS arrivi_totali,
           SUM(presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired
""").df()
print(verifica)

Righe escluse (comune non disponibile): 277
Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\arrivi_presenze_sired.csv
   n_righe  n_comuni  n_anni  arrivi_totali  presenze_totali
0   409461       330       4     17195421.0       73469531.0


In [18]:
# ============================================================
# FASE 1 - BLOCCO F: Armonizzazione e concatenazione Capacità Ricettiva (corretto)
# ============================================================
import pandas as pd

file_capacita_per_anno = {
    2022: RAW_DIR / "capacita_strutture_ricettive_annuale_2022.csv",
    2023: RAW_DIR / "capacita_strutture_ricettive_annuale_2023.csv",
    2024: RAW_DIR / "capacita_strutture_ricettive_annuale_2024.csv",
    2025: RAW_DIR / "capacita_strutture_ricettive_annuale_2025.csv",
}

def leggi_csv_con_fallback(path):
    try:
        return pd.read_csv(path, sep=',', encoding='utf-8')
    except UnicodeDecodeError:
        print(f"  -> {path.name}: non UTF-8, uso encoding cp1252")
        return pd.read_csv(path, sep=',', encoding='cp1252')

def carica_e_armonizza_capacita(path, anno):
    df = leggi_csv_con_fallback(path)
    
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.rename(columns={
        "stelle": "categoria",
        "numero strutture": "numero_strutture",
    })
    
    colonne_attese = ["anno", "provincia", "comune", "tipologia", "categoria", "numero_strutture", "letti", "camere"]
    mancanti = [c for c in colonne_attese if c not in df.columns]
    if mancanti:
        print(f"  ATTENZIONE anno {anno}: colonne mancanti rispetto allo schema comune: {mancanti}")
    
    df = df[[c for c in colonne_attese if c in df.columns]]
    
    df["comune"] = df["comune"].astype(str).str.strip().str.title()
    df["categoria"] = df["categoria"].replace("NULL", pd.NA)
    
    df["anno_verifica"] = anno
    return df

liste_df_capacita = []
for anno, path in file_capacita_per_anno.items():
    df_anno = carica_e_armonizza_capacita(path, anno)
    print(f"{anno}: {len(df_anno)} righe")
    liste_df_capacita.append(df_anno)

df_capacita_completo = pd.concat(liste_df_capacita, ignore_index=True)

disallineati = df_capacita_completo[df_capacita_completo["anno"].astype(str) != df_capacita_completo["anno_verifica"].astype(str)]
print(f"\nRighe con anno disallineato: {len(disallineati)}")
df_capacita_completo = df_capacita_completo.drop(columns=["anno_verifica"])

print(f"\nTotale righe concatenate: {len(df_capacita_completo)}")
print(f"Comuni distinti: {df_capacita_completo['comune'].nunique()}")
print(f"\nRiepilogo per anno:")
df_capacita_completo.groupby("anno")[["letti", "camere", "numero_strutture"]].sum()

2022: 1876 righe
  -> capacita_strutture_ricettive_annuale_2023.csv: non UTF-8, uso encoding cp1252
2023: 1914 righe
2024: 2328 righe
2025: 2471 righe

Righe con anno disallineato: 0

Totale righe concatenate: 8589
Comuni distinti: 354

Riepilogo per anno:


,letti,camere,numero_strutture
anno,,,
2022,302183,117261,21491
2023,324181,122209,27048
2024,388843,156536,39525
2025,459983,187897,52147


In [19]:
# ============================================================
# FASE 1 - BLOCCO G: Salvataggio Capacità Ricettiva in raw
# ============================================================

PATH_OUTPUT_CAPACITA = RAW_DIR / "capacita_ricettiva.csv"
df_capacita_completo.to_csv(PATH_OUTPUT_CAPACITA, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_CAPACITA}")

con.execute("DROP TABLE IF EXISTS raw.capacita_ricettiva")
con.execute(f"""
    CREATE TABLE raw.capacita_ricettiva AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_CAPACITA.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(letti) AS letti_totali
    FROM raw.capacita_ricettiva
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\capacita_ricettiva.csv
   n_righe  n_comuni  n_anni  letti_totali
0     8589       354       4     1475190.0


In [20]:
# ============================================================
# FASE 1 - BLOCCO J: Pulizia abitazioni occupate/non occupate (corretto)
# ============================================================
import pandas as pd

PATH_ABITAZIONI = RAW_DIR / "Abitazioni occupate e non occupate - comuni.csv"

df_abitazioni_raw = pd.read_csv(
    PATH_ABITAZIONI, 
    sep=',', 
    encoding='utf-8-sig',  # gestisce il BOM a inizio file
    quotechar="'"          # gestisce le note con virgole racchiuse in apici singoli
)

print(f"Righe totali (tutti i livelli territoriali d'Italia): {len(df_abitazioni_raw)}")
print(f"Valori distinti INDICATOR: {df_abitazioni_raw['INDICATOR'].unique()}")
print(f"Anni disponibili: {sorted(df_abitazioni_raw['TIME_PERIOD'].unique())}")

# Isoliamo solo le righe di livello comunale (REF_AREA puramente numerico)
df_abitazioni_raw["e_comune"] = df_abitazioni_raw["REF_AREA"].astype(str).str.isdigit()

df_solo_comuni = df_abitazioni_raw[df_abitazioni_raw["e_comune"]].copy()
print(f"\nRighe che sembrano essere di livello comunale: {len(df_solo_comuni)}")
print(f"Comuni distinti (tutta Italia): {df_solo_comuni['Territorio'].nunique()}")

Righe totali (tutti i livelli territoriali d'Italia): 71041
Valori distinti INDICATOR: <ArrowStringArray>
['NUM_OCC_DW_AV', 'NUM_UNOCC_DW_AV', 'NUM_DW_AV']
Length: 3, dtype: str
Anni disponibili: [np.int64(2019), np.int64(2021), np.int64(2023)]

Righe che sembrano essere di livello comunale: 69853
Comuni distinti (tutta Italia): 7774


In [21]:
# ============================================================
# FASE 1 - BLOCCO K: Filtro Sardegna + pivot indicatori
# ============================================================

# Uniforma il nome comune per il join (stesso trattamento delle altre fonti)
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio"].astype(str).str.strip().str.title()

# Lista comuni sardi dalla nostra anagrafica di riferimento
comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna)].copy()

print(f"Righe Sardegna: {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati nel file: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

# Comuni sardi NON trovati (utile per capire se il join ha problemi di naming)
comuni_mancanti = comuni_sardegna - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi non trovati nel file abitazioni: {len(comuni_mancanti)}")
if comuni_mancanti:
    print(sorted(comuni_mancanti))

# Trasformiamo da formato lungo a largo: una riga per comune-anno, colonne separate per indicatore
df_abitazioni_wide = df_abitazioni_sardegna.pivot_table(
    index=["comune_pulito", "TIME_PERIOD"],
    columns="INDICATOR",
    values="Osservazione",
    aggfunc="first"
).reset_index()

df_abitazioni_wide.columns.name = None
df_abitazioni_wide = df_abitazioni_wide.rename(columns={
    "comune_pulito": "comune",
    "TIME_PERIOD": "anno",
    "NUM_OCC_DW_AV": "abitazioni_occupate",
    "NUM_UNOCC_DW_AV": "abitazioni_non_occupate",
    "NUM_DW_AV": "abitazioni_totali"
})

# Controllo di coerenza: occupate + non occupate deve tornare vicino al totale
df_abitazioni_wide["somma_controllo"] = df_abitazioni_wide["abitazioni_occupate"] + df_abitazioni_wide["abitazioni_non_occupate"]
df_abitazioni_wide["differenza"] = df_abitazioni_wide["abitazioni_totali"] - df_abitazioni_wide["somma_controllo"]

print(f"\nComuni-anno nel formato finale: {len(df_abitazioni_wide)}")
print(f"Differenze totale vs occupate+non occupate (dovrebbero essere ~0):")
print(df_abitazioni_wide["differenza"].describe())

df_abitazioni_wide.head(10)

Righe Sardegna: 3330
Comuni sardi trovati nel file: 369 (su 377 attesi)

Comuni sardi non trovati nel file abitazioni: 8
["Quartu Sant'Elena", "San Nicolò D'Arcidano", "Sant'Andrea Frius", "Sant'Anna Arresi", "Sant'Antioco", "Sant'Antonio Di Gallura", "Trinità D'Agultu E Vignola", "Villa Sant'Antonio"]

Comuni-anno nel formato finale: 1107
Differenze totale vs occupate+non occupate (dovrebbero essere ~0):
count    1107.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: differenza, dtype: float64


,comune,anno,abitazioni_totali,abitazioni_occupate,abitazioni_non_occupate,somma_controllo,differenza
0,Abbasanta,2019,1609,1076,533,1609,0
1,Abbasanta,2021,1610,1128,482,1610,0
2,Abbasanta,2023,1635,1149,486,1635,0
3,Aggius,2019,1024,612,412,1024,0
4,Aggius,2021,1026,622,404,1026,0
5,Aggius,2023,1038,629,409,1038,0
6,Aglientu,2019,3255,655,2600,3255,0
7,Aglientu,2021,3264,674,2590,3264,0
8,Aglientu,2023,3293,701,2592,3293,0
9,Aidomaggiore,2019,354,192,162,354,0


In [22]:
# ============================================================
# FASE 1 - BLOCCO K-bis: Normalizzazione apostrofi e verifica
# ============================================================

def normalizza_apostrofi(testo):
    if pd.isna(testo):
        return testo
    return str(testo).replace("’", "'").replace("‘", "'")

# Applichiamo la normalizzazione a entrambe le fonti prima del confronto
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio"].apply(normalizza_apostrofi).str.strip().str.title()
comuni_sardegna_norm = set(df_anagrafica["comune"].apply(normalizza_apostrofi).str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna_norm)].copy()

print(f"Righe Sardegna (dopo normalizzazione apostrofi): {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

comuni_mancanti_bis = comuni_sardegna_norm - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi ancora non trovati: {len(comuni_mancanti_bis)}")
if comuni_mancanti_bis:
    print(sorted(comuni_mancanti_bis))

Righe Sardegna (dopo normalizzazione apostrofi): 3330
Comuni sardi trovati: 369 (su 377 attesi)

Comuni sardi ancora non trovati: 8
["Quartu Sant'Elena", "San Nicolò D'Arcidano", "Sant'Andrea Frius", "Sant'Anna Arresi", "Sant'Antioco", "Sant'Antonio Di Gallura", "Trinità D'Agultu E Vignola", "Villa Sant'Antonio"]


In [23]:
# ============================================================
# FASE 1 - BLOCCO K - ter
# ============================================================
import csv
import pandas as pd

# IMPORTANTE: usa il file CSV originale, quello scaricato la primissima volta
# da IstatData - NON il file .ods corretto a mano, NON un file già passato
# per pandas con quotechar="'"
PATH_ABITAZIONI = RAW_DIR / "Abitazioni occupate e non occupate - comuni.csv"

righe = []
with open(PATH_ABITAZIONI, encoding='utf-8-sig') as f:
    reader = csv.reader(f, delimiter=',', quoting=csv.QUOTE_NONE)
    header_completo = next(reader)
    colonne_necessarie = header_completo[:8]
    
    for riga in reader:
        righe.append(riga[:8])

df_abitazioni_raw = pd.DataFrame(righe, columns=colonne_necessarie)
df_abitazioni_raw["TIME_PERIOD"] = pd.to_numeric(df_abitazioni_raw["TIME_PERIOD"], errors="coerce")
df_abitazioni_raw["Osservazione"] = pd.to_numeric(df_abitazioni_raw["Osservazione"], errors="coerce")

# Solo comuni (REF_AREA numerico)
df_solo_comuni = df_abitazioni_raw[df_abitazioni_raw["REF_AREA"].astype(str).str.isdigit()].copy()

# Controllo apostrofi PRIMA di filtrare la Sardegna - deve essere pulito qui
controllo = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant'Antioco|Sant'Anna Arresi", case=False, na=False)]
print("Controllo apostrofi (deve essere pulito):")
for nome in controllo["Territorio"].unique():
    print(f"  {repr(nome)}")

Controllo apostrofi (deve essere pulito):


In [24]:
# ============================================================
# FASE 1 - BLOCCO K-quinto-bis: Diagnosi ampia su "Sant"
# ============================================================

righe_sant = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant", case=False, na=False)]
nomi_sant = sorted(righe_sant["Territorio"].unique())

print(f"Trovati {len(nomi_sant)} valori distinti con 'Sant':")
for nome in nomi_sant[:20]:
    print(f"  {repr(nome)}")

Trovati 206 valori distinti con 'Sant':
  '\'Aci Sant"\'Antonio\''
  '\'Albano Sant"\'Alessandro\''
  '\'Boschi Sant"\'Anna\''
  '\'Castel Sant"\'Angelo\''
  '\'Castel Sant"\'Elia\''
  '\'Castronuovo di Sant"\'Andrea\''
  '\'Cazzano Sant"\'Andrea\''
  '\'Città Sant"\'Angelo\''
  '\'Godega di Sant"\'Urbano\''
  '\'Isola Sant"\'Antonio\''
  '\'Mazzarrà Sant"\'Andrea\''
  '\'Monte Sant"\'Angelo\''
  '\'Mosciano Sant"\'Angelo\''
  '\'Motta Sant"\'Anastasia\''
  '\'Penna Sant"\'Andrea\''
  '\'Porto Sant"\'Elpidio\''
  '\'Quartu Sant"\'Elena\''
  '\'Rocchetta Sant"\'Antonio\''
  '\'Sant"\'Agapito\''
  '\'Sant"\'Agata Bolognese\''


In [25]:
# ============================================================
# FASE 1 - BLOCCO K-sesto: Correzione pattern apostrofo corrotto
# ============================================================
import re

def ripara_apostrofi(nome):
    if pd.isna(nome):
        return nome
    nome = str(nome)
    # Toglie un apostrofo "fantasma" a inizio e fine stringa, se presente
    if nome.startswith("'") and nome.endswith("'"):
        nome = nome[1:-1]
    # Sostituisce la sequenza corrotta "'  con un vero apostrofo
    nome = nome.replace('"\'', "'")
    return nome

df_solo_comuni["Territorio_riparato"] = df_solo_comuni["Territorio"].apply(ripara_apostrofi)

# Verifica sui casi noti
controllo = df_solo_comuni[df_solo_comuni["Territorio"].str.contains("Sant", case=False, na=False)]
print("Prima → Dopo la riparazione (primi 10 casi con 'Sant'):")
for _, row in controllo.head(10).iterrows():
    print(f"  {repr(row['Territorio'])}  →  {repr(row['Territorio_riparato'])}")

Prima → Dopo la riparazione (primi 10 casi con 'Sant'):
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Ambrogio di Torino\''  →  "Sant'Ambrogio di Torino"
  '\'Sant"\'Antonino di Susa\''  →  "Sant'Antonino di Susa"


In [26]:
# ============================================================
# FASE 1 - BLOCCO K-settimo: Filtro Sardegna + pivot (versione finale)
# ============================================================

# Usiamo la colonna riparata per il join
df_solo_comuni["comune_pulito"] = df_solo_comuni["Territorio_riparato"].astype(str).str.strip().str.title()

comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())

df_abitazioni_sardegna = df_solo_comuni[df_solo_comuni["comune_pulito"].isin(comuni_sardegna)].copy()

print(f"Righe Sardegna: {len(df_abitazioni_sardegna)}")
print(f"Comuni sardi trovati: {df_abitazioni_sardegna['comune_pulito'].nunique()} (su 377 attesi)")

comuni_mancanti = comuni_sardegna - set(df_abitazioni_sardegna["comune_pulito"].unique())
print(f"\nComuni sardi ancora non trovati: {len(comuni_mancanti)}")
if comuni_mancanti:
    print(sorted(comuni_mancanti))

# Pivot in formato largo
df_abitazioni_wide = df_abitazioni_sardegna.pivot_table(
    index=["comune_pulito", "TIME_PERIOD"],
    columns="INDICATOR",
    values="Osservazione",
    aggfunc="first"
).reset_index()

df_abitazioni_wide.columns.name = None
df_abitazioni_wide = df_abitazioni_wide.rename(columns={
    "comune_pulito": "comune",
    "TIME_PERIOD": "anno",
    "NUM_OCC_DW_AV": "abitazioni_occupate",
    "NUM_UNOCC_DW_AV": "abitazioni_non_occupate",
    "NUM_DW_AV": "abitazioni_totali"
})

df_abitazioni_wide["somma_controllo"] = df_abitazioni_wide["abitazioni_occupate"] + df_abitazioni_wide["abitazioni_non_occupate"]
df_abitazioni_wide["differenza"] = df_abitazioni_wide["abitazioni_totali"] - df_abitazioni_wide["somma_controllo"]

print(f"\nComuni-anno nel formato finale: {len(df_abitazioni_wide)}")
print(f"Controllo coerenza (dovrebbe essere tutto 0):")
print(df_abitazioni_wide["differenza"].describe())

Righe Sardegna: 3402
Comuni sardi trovati: 377 (su 377 attesi)

Comuni sardi ancora non trovati: 0

Comuni-anno nel formato finale: 1131
Controllo coerenza (dovrebbe essere tutto 0):
count    1131.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: differenza, dtype: float64


In [27]:
# ============================================================
# FASE 1 - BLOCCO K-ottavo: Salvataggio Abitazioni in raw
# ============================================================

PATH_OUTPUT_ABITAZIONI = RAW_DIR / "abitazioni_non_occupate.csv"
df_abitazioni_wide.to_csv(PATH_OUTPUT_ABITAZIONI, index=False, encoding='utf-8')
print(f"Salvato in: {PATH_OUTPUT_ABITAZIONI}")

con.execute("DROP TABLE IF EXISTS raw.abitazioni_non_occupate")
con.execute(f"""
    CREATE TABLE raw.abitazioni_non_occupate AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_ABITAZIONI.as_posix()}')
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe,
           COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni,
           SUM(abitazioni_non_occupate) AS totale_non_occupate_sardegna,
           SUM(abitazioni_occupate) AS totale_occupate_sardegna
    FROM raw.abitazioni_non_occupate
""").df()
print(verifica)

Salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\raw\abitazioni_non_occupate.csv
   n_righe  n_comuni  n_anni  totale_non_occupate_sardegna  \
0     1131       377       3                      922452.0   

   totale_occupate_sardegna  
0                 2157984.0  


In [28]:
# ============================================================
# FASE 1 - BLOCCO L: Popolazione residente (2022-2025) e Superficie
# ============================================================
import pandas as pd

# --- Popolazione: 4 file, uno per anno ---
file_popolazione_per_anno = {
    2022: RAW_DIR / "Residenti 2022_Sardegna_totali_comune.csv",
    2023: RAW_DIR / "Residenti 2023_Sardegna_totali_comune.csv",
    2024: RAW_DIR / "Residenti 2024_Sardegna_totali_comune.csv",
    2025: RAW_DIR / "Residenti 2025_Sardegna_totali_comune.csv",
}

liste_popolazione = []
for anno, path in file_popolazione_per_anno.items():
    df = pd.read_csv(path, sep=',', encoding='utf-8-sig')
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"Comune": "comune", "Totale": "popolazione_residente"})
    df["comune"] = df["comune"].str.strip().str.title()
    df["anno"] = anno
    liste_popolazione.append(df)

df_popolazione = pd.concat(liste_popolazione, ignore_index=True)
print(f"Popolazione: {len(df_popolazione)} righe, {df_popolazione['comune'].nunique()} comuni, anni: {sorted(df_popolazione['anno'].unique())}")

# --- Superficie: 1 solo file (costante nel tempo) ---
PATH_SUPERFICIE = RAW_DIR / "Superficie_2025_Sardegna_totali_comune.csv"

df_superficie = pd.read_csv(PATH_SUPERFICIE, sep=',', encoding='utf-8-sig')
df_superficie.columns = [c.strip() for c in df_superficie.columns]
df_superficie = df_superficie.rename(columns={"Comune": "comune", "Superficie_kmq": "superficie_kmq"})
df_superficie["comune"] = df_superficie["comune"].str.strip().str.title()

# Conversione decimale italiano (virgola) -> numero
df_superficie["superficie_kmq"] = (
    df_superficie["superficie_kmq"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print(f"\nSuperficie: {len(df_superficie)} righe, {df_superficie['comune'].nunique()} comuni")
print(f"Superficie totale Sardegna: {df_superficie['superficie_kmq'].sum():.1f} kmq")

# Controllo comuni mancanti rispetto all'anagrafica (stesso controllo fatto altre volte)
comuni_sardegna = set(df_anagrafica["comune"].str.strip().str.title())
mancanti_pop = comuni_sardegna - set(df_popolazione["comune"].unique())
mancanti_sup = comuni_sardegna - set(df_superficie["comune"].unique())
print(f"\nComuni mancanti in popolazione: {len(mancanti_pop)} -> {sorted(mancanti_pop)}")
print(f"Comuni mancanti in superficie: {len(mancanti_sup)} -> {sorted(mancanti_sup)}")

Popolazione: 1508 righe, 377 comuni, anni: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Superficie: 377 righe, 377 comuni
Superficie totale Sardegna: 24109.9 kmq

Comuni mancanti in popolazione: 0 -> []
Comuni mancanti in superficie: 0 -> []


In [29]:
# ============================================================
# FASE 1 - BLOCCO L-bis: Salvataggio Popolazione e Superficie in raw
# ============================================================

# Popolazione
PATH_OUTPUT_POP = RAW_DIR / "popolazione_residente.csv"
df_popolazione.to_csv(PATH_OUTPUT_POP, index=False, encoding='utf-8')

con.execute("DROP TABLE IF EXISTS raw.popolazione_residente")
con.execute(f"""
    CREATE TABLE raw.popolazione_residente AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_POP.as_posix()}')
""")

# Superficie
PATH_OUTPUT_SUP = RAW_DIR / "superficie_comunale.csv"
df_superficie.to_csv(PATH_OUTPUT_SUP, index=False, encoding='utf-8')

con.execute("DROP TABLE IF EXISTS raw.superficie_comunale")
con.execute(f"""
    CREATE TABLE raw.superficie_comunale AS 
    SELECT * FROM read_csv_auto('{PATH_OUTPUT_SUP.as_posix()}')
""")

verifica = con.execute("""
    SELECT 
        (SELECT COUNT(*) FROM raw.popolazione_residente) AS righe_popolazione,
        (SELECT COUNT(DISTINCT comune) FROM raw.popolazione_residente) AS comuni_popolazione,
        (SELECT COUNT(*) FROM raw.superficie_comunale) AS righe_superficie,
        (SELECT SUM(superficie_kmq) FROM raw.superficie_comunale) AS superficie_totale
""").df()
print(verifica)

   righe_popolazione  comuni_popolazione  righe_superficie  superficie_totale
0               1508                 377               377          24109.945


In [30]:
# ============================================================
# FASE 2 - STAGING-A: Tabella di riferimento comuni
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.comuni_riferimento AS
    SELECT 
        codice_istat_comune,
        comune,
        provincia_sigla,
        ripartizione_geografica
    FROM raw.anagrafica_comuni
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, COUNT(DISTINCT comune) AS n_nomi_unici
    FROM staging.comuni_riferimento
""").df()
print(verifica)

con.execute("SELECT * FROM staging.comuni_riferimento LIMIT 5").df()

   n_comuni  n_nomi_unici
0       377           377


,codice_istat_comune,comune,provincia_sigla,ripartizione_geografica
0,90001,Aggius,SS,Isole
1,90002,Alà dei Sardi,SS,Isole
2,90003,Alghero,SS,Isole
3,90004,Anela,SS,Isole
4,90005,Ardara,SS,Isole


In [31]:
# ============================================================
# FASE 2 - STAGING-B: Verifica comuni con apostrofo
# ============================================================

comuni_sospetti = ["Villa Sant'Antonio", "Sant'Antonio Di Gallura", "San Nicolò D'Arcidano", "Trinità D'Agultu E Vignola", "Domus De Maria"]

for nome in comuni_sospetti:
    parola_chiave = nome.split()[-1]  # ultima parola, di solito la più distintiva
    check = con.execute(
        "SELECT DISTINCT comune FROM raw.arrivi_presenze_sired WHERE comune LIKE ?",
        [f"%{parola_chiave}%"]
    ).df()
    print(f"Cercando '{nome}' (parola chiave: '{parola_chiave}'):")
    print(check)
    print()
    

Cercando 'Villa Sant'Antonio' (parola chiave: 'Sant'Antonio'):
                    comune
0  Sant'Antonio Di Gallura

Cercando 'Sant'Antonio Di Gallura' (parola chiave: 'Gallura'):
                    comune
0     Santa Teresa Gallura
1  Sant'Antonio Di Gallura

Cercando 'San Nicolò D'Arcidano' (parola chiave: 'D'Arcidano'):
                  comune
0  San Nicolò D'Arcidano

Cercando 'Trinità D'Agultu E Vignola' (parola chiave: 'Vignola'):
                       comune
0  Trinità D'Agultu E Vignola

Cercando 'Domus De Maria' (parola chiave: 'Maria'):
                 comune
0        Domus De Maria
1  Santa Maria Coghinas



In [32]:
# ============================================================
# FASE 2 - STAGING-A-bis: Chiave di join normalizzata
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.comuni_riferimento AS
    SELECT 
        codice_istat_comune,
        comune,
        provincia_sigla,
        ripartizione_geografica,
        LOWER(TRIM(REPLACE(comune, '’', ''''))) AS chiave_comune
    FROM raw.anagrafica_comuni
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_comuni, COUNT(DISTINCT chiave_comune) AS n_chiavi_uniche
    FROM staging.comuni_riferimento
""").df()
print(verifica)

   n_comuni  n_chiavi_uniche
0       377              377


In [33]:
# ============================================================
# FASE 2 - STAGING-B-bis: Arrivi/Presenze (con chiave normalizzata)
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.arrivi_presenze_annuale AS
    SELECT 
        r.comune,  -- nome ufficiale dall'anagrafica, non quello grezzo di SIRED
        a.anno,
        SUM(a.arrivi) AS arrivi_totali,
        SUM(a.presenze) AS presenze_totali
    FROM raw.arrivi_presenze_sired a
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(a.comune, '’', ''''))) = r.chiave_comune
    GROUP BY r.comune, a.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni,
           SUM(arrivi_totali) AS arrivi_totali_sardegna, SUM(presenze_totali) AS presenze_totali_sardegna
    FROM staging.arrivi_presenze_annuale
""").df()
print(verifica)

mancanti = con.execute("""
    SELECT r.comune
    FROM staging.comuni_riferimento r
    LEFT JOIN staging.arrivi_presenze_annuale a ON r.comune = a.comune
    WHERE a.comune IS NULL
""").df()
print(f"\nComuni senza dati arrivi/presenze: {len(mancanti)}")
print(mancanti['comune'].tolist())

   n_righe  n_comuni  n_anni  arrivi_totali_sardegna  presenze_totali_sardegna
0     1106       330       4              17195421.0                73469531.0

Comuni senza dati arrivi/presenze: 47
['Asuni', 'San Basilio', 'Ittireddu', 'Semestene', 'Olzai', 'Zerfaliu', 'Cargeghe', 'Neoneli', 'Bulzi', 'Noragugume', 'Barrali', 'Giave', 'Soddì', 'Ortacesus', 'Setzu', 'Illorai', "Villa Sant'Antonio", 'Montresta', 'Borutta', 'Tadasuni', 'Siapiccia', 'Birori', 'Oniferi', 'Simala', 'Segariu', 'Siris', 'Gesico', 'Siligo', 'Gonnoscodina', 'Morgongiori', 'Senis', 'Tiana', 'Orotelli', 'San Nicolò Gerrei', 'Suelli', 'Nule', 'Lei', 'Bidonì', 'Genuri', 'Ussaramanna', 'Esporlatu', 'Romana', 'Onanì', 'Mogorella', 'Villanova Truschedu', 'Curcuris', 'Goni']


In [34]:
# ============================================================
# FASE 2 - STAGING-C: Capacità ricettiva aggregata per comune×anno
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.capacita_annuale AS
    SELECT 
        r.comune,
        c.anno,
        SUM(c.numero_strutture) AS numero_strutture_totali,
        SUM(c.letti) AS letti_totali,
        SUM(c.camere) AS camere_totali
    FROM raw.capacita_ricettiva c
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(c.comune, '’', ''''))) = r.chiave_comune
    GROUP BY r.comune, c.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni,
           SUM(letti_totali) AS letti_totali_sardegna
    FROM staging.capacita_annuale
""").df()
print(verifica)

mancanti_capacita = con.execute("""
    SELECT r.comune
    FROM staging.comuni_riferimento r
    LEFT JOIN staging.capacita_annuale c ON r.comune = c.comune
    WHERE c.comune IS NULL
""").df()
print(f"\nComuni senza dati capacità ricettiva: {len(mancanti_capacita)}")

   n_righe  n_comuni  n_anni  letti_totali_sardegna
0     1351       354       4              1475190.0

Comuni senza dati capacità ricettiva: 23


In [35]:
# ============================================================
# FASE 2 - STAGING-D: Popolazione, Superficie, Abitazioni
# ============================================================

# --- Popolazione ---
con.execute("""
    CREATE OR REPLACE TABLE staging.popolazione AS
    SELECT 
        r.comune,
        p.anno,
        p.popolazione_residente
    FROM raw.popolazione_residente p
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(p.comune, '’', ''''))) = r.chiave_comune
""")

# --- Superficie (nessun anno, costante) ---
con.execute("""
    CREATE OR REPLACE TABLE staging.superficie AS
    SELECT 
        r.comune,
        s.superficie_kmq
    FROM raw.superficie_comunale s
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(s.comune, '’', ''''))) = r.chiave_comune
""")

# --- Abitazioni non occupate ---
con.execute("""
    CREATE OR REPLACE TABLE staging.abitazioni AS
    SELECT 
        r.comune,
        CAST(a.anno AS INTEGER) AS anno,
        a.abitazioni_occupate,
        a.abitazioni_non_occupate,
        a.abitazioni_totali
    FROM raw.abitazioni_non_occupate a
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(a.comune, '’', ''''))) = r.chiave_comune
""")

# Verifica delle tre insieme
for tabella in ["popolazione", "superficie", "abitazioni"]:
    v = con.execute(f"SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni FROM staging.{tabella}").df()
    print(f"{tabella}: {v.iloc[0]['n_righe']} righe, {v.iloc[0]['n_comuni']} comuni")

popolazione: 1508 righe, 377 comuni
superficie: 377 righe, 377 comuni
abitazioni: 1131 righe, 377 comuni


In [36]:
# ============================================================
# FASE 2 - STAGING-F: Join finale - tabella presentation
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_ultimo_disponibile AS (
        -- Per ogni comune, usiamo il dato 2023 (l'ultimo disponibile) per gli anni 2024/2025
        SELECT comune, abitazioni_occupate, abitazioni_non_occupate, abitazioni_totali
        FROM staging.abitazioni
        WHERE anno = 2023
    )
    SELECT 
        g.comune,
        g.anno,
        p.popolazione_residente,
        s.superficie_kmq,
        ap.arrivi_totali,
        ap.presenze_totali,
        c.numero_strutture_totali,
        c.letti_totali,
        c.camere_totali,
        ab.abitazioni_occupate,
        ab.abitazioni_non_occupate,
        ab.abitazioni_totali
    FROM griglia g
    LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
    LEFT JOIN staging.superficie s ON g.comune = s.comune
    LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
    LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
    LEFT JOIN abitazioni_ultimo_disponibile ab ON g.comune = ab.comune
    ORDER BY g.comune, g.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, COUNT(DISTINCT anno) AS n_anni
    FROM presentation.indicatori_comune_anno
""").df()
print(verifica)

# Controllo completezza per colonna (quanti NULL per ciascuna variabile)
completezza = con.execute("""
    SELECT 
        SUM(CASE WHEN popolazione_residente IS NULL THEN 1 ELSE 0 END) AS null_popolazione,
        SUM(CASE WHEN superficie_kmq IS NULL THEN 1 ELSE 0 END) AS null_superficie,
        SUM(CASE WHEN arrivi_totali IS NULL THEN 1 ELSE 0 END) AS null_arrivi,
        SUM(CASE WHEN letti_totali IS NULL THEN 1 ELSE 0 END) AS null_letti,
        SUM(CASE WHEN abitazioni_non_occupate IS NULL THEN 1 ELSE 0 END) AS null_abitazioni
    FROM presentation.indicatori_comune_anno
""").df()
print(completezza)

con.execute("SELECT * FROM presentation.indicatori_comune_anno WHERE comune = 'Villasimius' ORDER BY anno").df()

   n_righe  n_comuni  n_anni
0     1508       377       4


   null_popolazione  null_superficie  null_arrivi  null_letti  null_abitazioni
0               0.0              0.0        402.0       157.0              0.0


,comune,anno,popolazione_residente,superficie_kmq,arrivi_totali,presenze_totali,numero_strutture_totali,letti_totali,camere_totali,abitazioni_occupate,abitazioni_non_occupate,abitazioni_totali
0,Villasimius,2022,3705,58.173,137933.0,753266.0,820.0,11887.0,4522.0,1941,4541,6482
1,Villasimius,2023,3689,58.173,133429.0,737623.0,827.0,11915.0,4503.0,1941,4541,6482
2,Villasimius,2024,3721,58.173,153737.0,824403.0,1290.0,14333.0,5530.0,1941,4541,6482
3,Villasimius,2025,3735,58.173,172783.0,916156.0,1793.0,17027.0,6765.0,1941,4541,6482


In [37]:
# ============================================================
# FASE 2 - STAGING-G: Indicatori derivati
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    SELECT 
        *,
        ROUND(presenze_totali / NULLIF(popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(presenze_totali / NULLIF(superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(presenze_totali / NULLIF(letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        ROUND(abitazioni_non_occupate * 100.0 / NULLIF(abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno
""")

con.execute("""
    SELECT comune, anno, presenze_per_residente, presenze_per_kmq, 
           tasso_occupazione_media, quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' 
    ORDER BY anno
""").df()

,comune,anno,presenze_per_residente,presenze_per_kmq,tasso_occupazione_media,quota_abitazioni_non_occupate_pct
0,Villasimius,2022,203.31,12948.7,0.174,70.1
1,Villasimius,2023,199.95,12679.8,0.170,70.1
2,Villasimius,2024,221.55,14171.6,0.158,70.1
3,Villasimius,2025,245.29,15748.8,0.147,70.1


In [38]:
# ============================================================
# FASE 2 - STAGING-F-bis: Abitazioni con anno più vicino per periodo
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_per_periodo AS (
        -- 2022 usa il dato censimento 2021 (il più vicino disponibile prima)
        -- 2023, 2024, 2025 usano il dato censimento 2023 (il più recente disponibile)
        SELECT 
            g.comune, 
            g.anno,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.abitazioni ab 
            ON g.comune = ab.comune 
            AND ab.anno = CASE WHEN g.anno = 2022 THEN 2021 ELSE 2023 END
    )
    SELECT 
        g.comune,
        g.anno,
        p.popolazione_residente,
        s.superficie_kmq,
        ap.arrivi_totali,
        ap.presenze_totali,
        c.numero_strutture_totali,
        c.letti_totali,
        c.camere_totali,
        ab.abitazioni_occupate,
        ab.abitazioni_non_occupate,
        ab.abitazioni_totali,
        ROUND(ap.presenze_totali / NULLIF(p.popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(ap.presenze_totali / NULLIF(s.superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(ap.presenze_totali / NULLIF(c.letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        ROUND(ab.abitazioni_non_occupate * 100.0 / NULLIF(ab.abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct
    FROM griglia g
    LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
    LEFT JOIN staging.superficie s ON g.comune = s.comune
    LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
    LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
    LEFT JOIN abitazioni_per_periodo ab ON g.comune = ab.comune AND g.anno = ab.anno
    ORDER BY g.comune, g.anno
""")

con.execute("""
    SELECT comune, anno, abitazioni_non_occupate, quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' 
    ORDER BY anno
""").df()

,comune,anno,abitazioni_non_occupate,quota_abitazioni_non_occupate_pct
0,Villasimius,2022,4505,70.4
1,Villasimius,2023,4541,70.1
2,Villasimius,2024,4541,70.1
3,Villasimius,2025,4541,70.1


In [39]:
# ============================================================
# FASE 2 - STAGING-F-ter: Tabella completa con tutti gli indicatori
# ============================================================

# Coefficiente per stimare i posti letto "informali" dalle abitazioni non occupate
# Assunzione dichiarata: numero medio di persone per abitazione in Italia (dato ISTAT recente ~2.3)
COEFFICIENTE_PERSONE_PER_ABITAZIONE = 2.3

con.execute(f"""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    WITH griglia AS (
        SELECT r.comune, anni.anno
        FROM staging.comuni_riferimento r
        CROSS JOIN (SELECT UNNEST([2022, 2023, 2024, 2025]) AS anno) anni
    ),
    abitazioni_per_periodo AS (
        SELECT 
            g.comune, 
            g.anno,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.abitazioni ab 
            ON g.comune = ab.comune 
            AND ab.anno = CASE WHEN g.anno = 2022 THEN 2021 ELSE 2023 END
    ),
    base AS (
        SELECT 
            g.comune,
            g.anno,
            p.popolazione_residente,
            s.superficie_kmq,
            ap.arrivi_totali,
            ap.presenze_totali,
            c.numero_strutture_totali,
            c.letti_totali,
            c.camere_totali,
            ab.abitazioni_occupate,
            ab.abitazioni_non_occupate,
            ab.abitazioni_totali
        FROM griglia g
        LEFT JOIN staging.popolazione p ON g.comune = p.comune AND g.anno = p.anno
        LEFT JOIN staging.superficie s ON g.comune = s.comune
        LEFT JOIN staging.arrivi_presenze_annuale ap ON g.comune = ap.comune AND g.anno = ap.anno
        LEFT JOIN staging.capacita_annuale c ON g.comune = c.comune AND g.anno = c.anno
        LEFT JOIN abitazioni_per_periodo ab ON g.comune = ab.comune AND g.anno = ab.anno
    ),
    base_con_lag AS (
        SELECT 
            *,
            LAG(presenze_totali) OVER (PARTITION BY comune ORDER BY anno) AS presenze_anno_precedente,
            LAG(letti_totali) OVER (PARTITION BY comune ORDER BY anno) AS letti_anno_precedente
        FROM base
    )
    SELECT 
        * EXCLUDE (presenze_anno_precedente, letti_anno_precedente),
        
        -- Pressione turistica di base
        ROUND(presenze_totali / NULLIF(popolazione_residente, 0), 2) AS presenze_per_residente,
        ROUND(presenze_totali / NULLIF(superficie_kmq, 0), 1) AS presenze_per_kmq,
        ROUND(presenze_totali / NULLIF(letti_totali * 365.0, 0), 3) AS tasso_occupazione_media,
        
        -- Comportamento turistico
        ROUND(presenze_totali / NULLIF(arrivi_totali, 0), 2) AS permanenza_media_giorni,
        
        -- Crescita anno su anno
        ROUND((presenze_totali - presenze_anno_precedente) * 100.0 / NULLIF(presenze_anno_precedente, 0), 1) AS crescita_presenze_yoy_pct,
        ROUND((letti_totali - letti_anno_precedente) * 100.0 / NULLIF(letti_anno_precedente, 0), 1) AS crescita_letti_yoy_pct,
        
        -- Struttura dell'offerta ricettiva
        ROUND(letti_totali / NULLIF(numero_strutture_totali, 0), 1) AS letti_per_struttura,
        ROUND(letti_totali / NULLIF(popolazione_residente, 0), 3) AS letti_per_residente,
        
        -- Seconde case / offerta informale
        ROUND(abitazioni_non_occupate * 100.0 / NULLIF(abitazioni_totali, 0), 1) AS quota_abitazioni_non_occupate_pct,
        ROUND(abitazioni_non_occupate * {COEFFICIENTE_PERSONE_PER_ABITAZIONE}, 0) AS posti_letto_informali_stimati,
        ROUND((letti_totali + abitazioni_non_occupate * {COEFFICIENTE_PERSONE_PER_ABITAZIONE}) / NULLIF(popolazione_residente, 0), 3) AS pressione_potenziale_totale,
        
        -- Contesto demografico
        ROUND(popolazione_residente / NULLIF(superficie_kmq, 0), 1) AS densita_abitanti_kmq
        
    FROM base_con_lag
    ORDER BY comune, anno
""")

verifica = con.execute("SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni FROM presentation.indicatori_comune_anno").df()
print(verifica)

con.execute("""
    SELECT * FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Villasimius' ORDER BY anno
""").df()

   n_righe  n_comuni
0     1508       377


,comune,anno,popolazione_residente,superficie_kmq,arrivi_totali,presenze_totali,numero_strutture_totali,letti_totali,camere_totali,abitazioni_occupate,...,tasso_occupazione_media,permanenza_media_giorni,crescita_presenze_yoy_pct,crescita_letti_yoy_pct,letti_per_struttura,letti_per_residente,quota_abitazioni_non_occupate_pct,posti_letto_informali_stimati,pressione_potenziale_totale,densita_abitanti_kmq
0,Villasimius,2022,3705,58.173,137933.0,753266.0,820.0,11887.0,4522.0,1893,...,0.174,5.46,NaN,NaN,14.5,3.208,70.4,10362.0,6.005,63.7
1,Villasimius,2023,3689,58.173,133429.0,737623.0,827.0,11915.0,4503.0,1941,...,0.170,5.53,-2.1,0.2,14.4,3.230,70.1,10444.0,6.061,63.4
2,Villasimius,2024,3721,58.173,153737.0,824403.0,1290.0,14333.0,5530.0,1941,...,0.158,5.36,11.8,20.3,11.1,3.852,70.1,10444.0,6.659,64.0
3,Villasimius,2025,3735,58.173,172783.0,916156.0,1793.0,17027.0,6765.0,1941,...,0.147,5.30,11.1,18.8,9.5,4.559,70.1,10444.0,7.355,64.2


In [40]:
# ============================================================
# FASE 2 - EXPORT: Tabella finale per Tableau
# ============================================================
import os

PRESENTATION_DIR = BASE_DIR / "data" / "presentation"
os.makedirs(PRESENTATION_DIR, exist_ok=True)

df_finale = con.execute("SELECT * FROM presentation.indicatori_comune_anno").df()

PATH_EXPORT_CSV = PRESENTATION_DIR / "indicatori_comune_anno.csv"
df_finale.to_csv(PATH_EXPORT_CSV, index=False, encoding='utf-8')
print(f"Esportato in: {PATH_EXPORT_CSV}")
print(f"Righe: {len(df_finale)}, Colonne: {len(df_finale.columns)}")

Esportato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\presentation\indicatori_comune_anno.csv
Righe: 1508, Colonne: 24


In [41]:
# ============================================================
# FASE 2 - STAGING-H: Presenze mensili reali per comune (corretto)
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.presenze_mensili AS
    SELECT 
        r.comune,
        a.anno,
        CAST(a.mese AS INTEGER) AS mese,
        SUM(a.presenze) AS presenze_mese
    FROM raw.arrivi_presenze_sired a
    JOIN staging.comuni_riferimento r 
        ON LOWER(TRIM(REPLACE(a.comune, '’', ''''))) = r.chiave_comune
    WHERE LOWER(TRIM(a.mese)) != 'non disponibile'
    GROUP BY r.comune, a.anno, CAST(a.mese AS INTEGER)
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni, 
           COUNT(DISTINCT anno) AS n_anni, COUNT(DISTINCT mese) AS n_mesi
    FROM staging.presenze_mensili
""").df()
print(verifica)

con.execute("""
    SELECT comune, anno, mese, presenze_mese 
    FROM staging.presenze_mensili 
    WHERE comune = 'Cagliari' AND anno = 2025
    ORDER BY mese
""").df()

   n_righe  n_comuni  n_anni  n_mesi
0     9727       322       4      12


,comune,anno,mese,presenze_mese
0,Cagliari,2025,1,35523.0
1,Cagliari,2025,2,39978.0
2,Cagliari,2025,3,49804.0
3,Cagliari,2025,4,92205.0
4,Cagliari,2025,5,111005.0
5,Cagliari,2025,6,125171.0
6,Cagliari,2025,7,154003.0
7,Cagliari,2025,8,175651.0
8,Cagliari,2025,9,140679.0
9,Cagliari,2025,10,105291.0


In [42]:
# ============================================================
# FASE 2 - STAGING-I: Calcolo Gini di stagionalità per comune-anno
# ============================================================
import numpy as np
import pandas as pd

df_mensile = con.execute("SELECT * FROM staging.presenze_mensili").df()

def gini_stagionalita(valori_mensili):
    x = np.sort(np.array(valori_mensili, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return None
    cum_x = np.cumsum(x)
    gini = (2 * np.sum(np.arange(1, n + 1) * x)) / (n * cum_x[-1]) - (n + 1) / n
    return round(gini, 4)

# Calcolo Gini per ogni comune-anno (serve avere tutti e 12 i mesi, altrimenti il Gini è distorto)
risultati_gini = []
for (comune, anno), gruppo in df_mensile.groupby(["comune", "anno"]):
    n_mesi_presenti = gruppo["mese"].nunique()
    valori = gruppo.set_index("mese")["presenze_mese"].reindex(range(1, 13), fill_value=0).values
    gini = gini_stagionalita(valori)
    risultati_gini.append({
        "comune": comune, 
        "anno": anno, 
        "gini_stagionalita": gini,
        "n_mesi_con_dati": n_mesi_presenti
    })

df_gini = pd.DataFrame(risultati_gini)

print(f"Comuni-anno calcolati: {len(df_gini)}")
print(f"\nDistribuzione n_mesi_con_dati (quanti comuni hanno tutti e 12 i mesi vs meno):")
print(df_gini["n_mesi_con_dati"].value_counts().sort_index())

print(f"\nTop 10 comuni per stagionalità più estrema (Gini più alto), anno 2025:")
print(df_gini[df_gini["anno"] == 2025].nlargest(10, "gini_stagionalita")[["comune", "gini_stagionalita"]])

print(f"\nComuni con stagionalità più bassa (Gini più basso), anno 2025:")
print(df_gini[df_gini["anno"] == 2025].nsmallest(10, "gini_stagionalita")[["comune", "gini_stagionalita"]])

Comuni-anno calcolati: 1074

Distribuzione n_mesi_con_dati (quanti comuni hanno tutti e 12 i mesi vs meno):
n_mesi_con_dati
1      37
2      50
3      37
4      34
5      50
6      45
7      72
8      63
9      60
10     72
11     93
12    461
Name: count, dtype: int64

Top 10 comuni per stagionalità più estrema (Gini più alto), anno 2025:
              comune  gini_stagionalita
23             Allai             0.9167
128        Boroneddu             0.9167
243          Dualchi             0.9167
579           Osidda             0.9167
620              Pau             0.9167
827             Seui             0.9167
992          Usellus             0.9167
1044  Villanova Tulo             0.9167
1047  Villanovaforru             0.9167
1059      Villasalto             0.9167

Comuni con stagionalità più bassa (Gini più basso), anno 2025:
         comune  gini_stagionalita
1006        Uta             0.0392
414     Macomer             0.0524
764     Sardara             0.1145
507       Nuor

In [43]:
# ============================================================
# FASE 2 - STAGING-I-bis: Top/Bottom Gini, solo comuni con copertura sufficiente
# ============================================================

SOGLIA_MESI_MINIMI = 10  # almeno 10 mesi su 12 con dati, per considerare il Gini affidabile

df_gini_affidabile = df_gini[df_gini["n_mesi_con_dati"] >= SOGLIA_MESI_MINIMI].copy()

print(f"Comuni-anno con copertura sufficiente (>= {SOGLIA_MESI_MINIMI} mesi): {len(df_gini_affidabile)} su {len(df_gini)}")

print(f"\nTop 10 stagionalità più estrema (2025, solo dati affidabili):")
print(df_gini_affidabile[df_gini_affidabile["anno"] == 2025].nlargest(10,"gini_stagionalita")[["comune", "gini_stagionalita", "n_mesi_con_dati"]])

print(f"\nBottom 10 stagionalità più bassa (2025, solo dati affidabili):")
print(df_gini_affidabile[df_gini_affidabile["anno"] == 2025].nsmallest(10,"gini_stagionalita")[["comune", "gini_stagionalita", "n_mesi_con_dati"]])

Comuni-anno con copertura sufficiente (>= 10 mesi): 626 su 1074

Top 10 stagionalità più estrema (2025, solo dati affidabili):
                         comune  gini_stagionalita  n_mesi_con_dati
11                     Aglientu             0.6826               10
935                       Torpè             0.6815               12
896                       Telti             0.6755               10
857                   Siniscola             0.6691               12
942                     Tortolì             0.6664               12
64                       Badesi             0.6662               12
885                    Stintino             0.6593               12
962  Trinità d'Agultu e Vignola             0.6507               12
918                     Teulada             0.6500               12
733            Sant'Anna Arresi             0.6495               12

Bottom 10 stagionalità più bassa (2025, solo dati affidabili):
         comune  gini_stagionalita  n_mesi_con_dati
1006     

In [44]:
# ============================================================
# FASE 2 - STAGING-I-ter: Doppio Gini (zero-filled vs solo mesi attivi)
# ============================================================

def gini_stagionalita(valori_mensili):
    x = np.sort(np.array(valori_mensili, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return None
    cum_x = np.cumsum(x)
    gini = (2 * np.sum(np.arange(1, n + 1) * x)) / (n * cum_x[-1]) - (n + 1) / n
    return round(gini, 4)

risultati_gini = []
for (comune, anno), gruppo in df_mensile.groupby(["comune", "anno"]):
    n_mesi_presenti = gruppo["mese"].nunique()
    
    # Versione A: mesi mancanti = zero (assume vera assenza di turismo)
    valori_zero_filled = gruppo.set_index("mese")["presenze_mese"].reindex(range(1, 13), fill_value=0).values
    gini_zero_filled = gini_stagionalita(valori_zero_filled)
    
    # Versione B: solo sui mesi che hanno davvero un dato (ignora quelli mancanti)
    valori_solo_attivi = gruppo["presenze_mese"].values
    gini_solo_attivi = gini_stagionalita(valori_solo_attivi) if n_mesi_presenti >= 3 else None
    
    risultati_gini.append({
        "comune": comune, "anno": anno,
        "gini_zero_filled": gini_zero_filled,
        "gini_solo_mesi_attivi": gini_solo_attivi,
        "n_mesi_con_dati": n_mesi_presenti
    })

df_gini = pd.DataFrame(risultati_gini)
print(df_gini[df_gini["anno"]==2025].nlargest(10, "gini_zero_filled")[["comune","gini_zero_filled","gini_solo_mesi_attivi","n_mesi_con_dati"]])

              comune  gini_zero_filled  gini_solo_mesi_attivi  n_mesi_con_dati
23             Allai            0.9167                    NaN                1
128        Boroneddu            0.9167                    NaN                1
243          Dualchi            0.9167                    NaN                1
579           Osidda            0.9167                    NaN                1
620              Pau            0.9167                    NaN                1
827             Seui            0.9167                    NaN                1
992          Usellus            0.9167                    NaN                1
1044  Villanova Tulo            0.9167                    NaN                1
1047  Villanovaforru            0.9167                    NaN                1
1059      Villasalto            0.9167                    NaN                1


In [45]:
# ============================================================
# FASE 2 - STAGING-I-quater: Salvataggio Gini e join finale
# ============================================================

con.execute("DROP TABLE IF EXISTS staging.gini_stagionalita")
con.register("df_gini_temp", df_gini)
con.execute("CREATE TABLE staging.gini_stagionalita AS SELECT * FROM df_gini_temp")

# Aggiungiamo le colonne Gini alla tabella presentation, senza perdere nulla di quello che c'è già
con.execute("""
    CREATE OR REPLACE TABLE presentation.indicatori_comune_anno AS
    SELECT 
        base.*,
        g.gini_zero_filled AS gini_stagionalita,
        g.gini_solo_mesi_attivi AS gini_stagionalita_solo_mesi_attivi,
        g.n_mesi_con_dati AS n_mesi_con_dati_turismo
    FROM presentation.indicatori_comune_anno base
    LEFT JOIN staging.gini_stagionalita g 
        ON base.comune = g.comune AND base.anno = g.anno
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni,
           COUNT(gini_stagionalita) AS n_con_gini
    FROM presentation.indicatori_comune_anno
""").df()
print(verifica)

con.execute("""
    SELECT comune, anno, gini_stagionalita, gini_stagionalita_solo_mesi_attivi, n_mesi_con_dati_turismo
    FROM presentation.indicatori_comune_anno 
    WHERE comune = 'Cagliari' 
    ORDER BY anno
""").df()

   n_righe  n_comuni  n_con_gini
0     1508       377        1074


,comune,anno,gini_stagionalita,gini_stagionalita_solo_mesi_attivi,n_mesi_con_dati_turismo
0,Cagliari,2022,0.2846,0.2846,12
1,Cagliari,2023,0.2719,0.2719,12
2,Cagliari,2024,0.2866,0.2866,12
3,Cagliari,2025,0.2827,0.2827,12


In [46]:
# ============================================================
# FASE 2 - EXPORT: Tabella aggiornata con Gini per Tableau
# ============================================================

df_finale = con.execute("SELECT * FROM presentation.indicatori_comune_anno").df()

PATH_EXPORT_CSV = PRESENTATION_DIR / "indicatori_comune_anno.csv"
df_finale.to_csv(PATH_EXPORT_CSV, index=False, encoding='utf-8')
print(f"Esportato in: {PATH_EXPORT_CSV}")
print(f"Righe: {len(df_finale)}, Colonne: {len(df_finale.columns)}")

Esportato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\presentation\indicatori_comune_anno.csv
Righe: 1508, Colonne: 27


In [47]:
# ============================================================
# FASE 2 - STAGING-J: Flusso regionale giornaliero (contesto, non comunale)
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.flusso_regionale_giornaliero AS
    SELECT 
        data,
        SUM(CASE WHEN "porto/aeroporto" = 'Aeroporto' THEN arrivi ELSE 0 END) AS arrivi_aeroporti,
        SUM(CASE WHEN "porto/aeroporto" = 'Porto' THEN arrivi ELSE 0 END) AS arrivi_porti,
        SUM(arrivi) AS arrivi_totali_sardegna,
        SUM(partenze) AS partenze_totali_sardegna
    FROM raw.porti_aeroporti
    GROUP BY data
    ORDER BY data
""")

verifica = con.execute("""
    SELECT COUNT(*) AS n_giorni, MIN(data) AS data_min, MAX(data) AS data_max,
           SUM(arrivi_totali_sardegna) AS arrivi_totali_periodo
    FROM staging.flusso_regionale_giornaliero
""").df()
print(verifica)

con.execute("""
    SELECT * FROM staging.flusso_regionale_giornaliero 
    WHERE data BETWEEN '2025-08-01' AND '2025-08-10'
    ORDER BY data
""").df()

   n_giorni   data_min   data_max  arrivi_totali_periodo
0      1651 2022-01-01 2026-07-11             33744225.0


,data,arrivi_aeroporti,arrivi_porti,arrivi_totali_sardegna,partenze_totali_sardegna
0,2025-08-01,31312.0,25291.0,56603.0,46188.0
1,2025-08-02,32934.0,32024.0,64958.0,56212.0
2,2025-08-03,31233.0,25808.0,57041.0,46792.0
3,2025-08-04,27941.0,21457.0,49398.0,41611.0
4,2025-08-05,26711.0,29033.0,55744.0,37000.0
5,2025-08-06,27717.0,20360.0,48077.0,34838.0
6,2025-08-07,26042.0,22039.0,48081.0,36280.0
7,2025-08-08,30968.0,27785.0,58753.0,45250.0
8,2025-08-09,32411.0,33374.0,65785.0,53917.0
9,2025-08-10,29891.0,28571.0,58462.0,48507.0


In [48]:
# ============================================================
# FASE 2 - STAGING-K: Flusso non catturato dal censimento ufficiale
# ============================================================

con.execute("""
    CREATE OR REPLACE TABLE staging.flusso_non_catturato_annuale AS
    WITH porti_aeroporti_annuale AS (
        SELECT 
            EXTRACT(YEAR FROM data) AS anno,
            SUM(arrivi) AS arrivi_porti_aeroporti
        FROM raw.porti_aeroporti
        GROUP BY EXTRACT(YEAR FROM data)
    ),
    sired_annuale AS (
        SELECT anno, SUM(arrivi_totali) AS arrivi_sired
        FROM staging.arrivi_presenze_annuale
        GROUP BY anno
    )
    SELECT 
        p.anno,
        p.arrivi_porti_aeroporti,
        s.arrivi_sired,
        p.arrivi_porti_aeroporti - s.arrivi_sired AS flusso_non_catturato,
        ROUND((p.arrivi_porti_aeroporti - s.arrivi_sired) * 100.0 / NULLIF(p.arrivi_porti_aeroporti, 0), 1) AS quota_non_catturata_pct
    FROM porti_aeroporti_annuale p
    JOIN sired_annuale s ON p.anno = s.anno
    ORDER BY p.anno
""")

con.execute("SELECT * FROM staging.flusso_non_catturato_annuale").df()

,anno,arrivi_porti_aeroporti,arrivi_sired,flusso_non_catturato,quota_non_catturata_pct
0,2022,6910435.0,3678902.0,3231533.0,46.8
1,2023,7285690.0,3904361.0,3381329.0,46.4
2,2024,7734717.0,4442089.0,3292628.0,42.6
3,2025,8138434.0,5170069.0,2968365.0,36.5


In [49]:
# ============================================================
# FASE 2 - EXPORT: Tabelle di contesto regionale per Tableau
# ============================================================

df_flusso_giornaliero = con.execute("SELECT * FROM staging.flusso_regionale_giornaliero").df()
df_flusso_giornaliero.to_csv(PRESENTATION_DIR / "flusso_regionale_giornaliero.csv", index=False, encoding='utf-8')

df_flusso_non_catturato = con.execute("SELECT * FROM staging.flusso_non_catturato_annuale").df()
df_flusso_non_catturato.to_csv(PRESENTATION_DIR / "flusso_non_catturato_annuale.csv", index=False, encoding='utf-8')

print("Esportate entrambe le tabelle di contesto regionale in data/presentation/")

Esportate entrambe le tabelle di contesto regionale in data/presentation/


In [50]:
# ============================================================
# CATALOGO DATI - Blocco A: Estrazione automatica struttura
# ============================================================

df_struttura = con.execute("""
    SELECT 
        table_schema,
        table_name,
        column_name,
        data_type,
        ordinal_position
    FROM information_schema.columns
    WHERE table_schema IN ('raw', 'staging', 'presentation')
    ORDER BY table_schema, table_name, ordinal_position
""").df()

print(f"Totale colonne censite: {len(df_struttura)}")
print(f"\nTabelle per schema:")
print(df_struttura.groupby("table_schema")["table_name"].nunique())
print(f"\nElenco tabelle:")
for schema in ["raw", "staging", "presentation"]:
    tabelle = df_struttura[df_struttura["table_schema"] == schema]["table_name"].unique()
    print(f"  {schema}: {list(tabelle)}")

Totale colonne censite: 109

Tabelle per schema:
table_schema
presentation     1
raw              7
staging         10
Name: table_name, dtype: int64

Elenco tabelle:
  raw: ['abitazioni_non_occupate', 'anagrafica_comuni', 'arrivi_presenze_sired', 'capacita_ricettiva', 'popolazione_residente', 'porti_aeroporti', 'superficie_comunale']
  staging: ['abitazioni', 'arrivi_presenze_annuale', 'capacita_annuale', 'comuni_riferimento', 'flusso_non_catturato_annuale', 'flusso_regionale_giornaliero', 'gini_stagionalita', 'popolazione', 'presenze_mensili', 'superficie']
  presentation: ['indicatori_comune_anno']


In [51]:
# ============================================================
# CATALOGO DATI - Blocco B: Dizionario descrizioni colonne
# ============================================================

# Mappa colonna -> (descrizione, unità di misura)
descrizioni_colonne = {
    # --- Chiavi e anagrafica ---
    "codice_istat_comune": ("Codice ISTAT alfanumerico del comune", "codice"),
    "codice_istat_numerico": ("Codice ISTAT numerico del comune", "codice"),
    "comune": ("Nome del comune, forma canonica (Title Case) da anagrafica ISTAT", "testo"),
    "provincia_sigla": ("Sigla della provincia (CA, SS, NU, OR, SU)", "testo"),
    "provincia": ("Nome provincia/città metropolitana (fonte originale, non normalizzato)", "testo"),
    "ripartizione_geografica": ("Ripartizione geografica ISTAT (sempre 'Isole' per la Sardegna)", "testo"),
    "chiave_comune": ("Chiave normalizzata (minuscolo, apostrofi uniformati) usata SOLO per i join tra fonti diverse, non per la visualizzazione", "testo"),
    "anno": ("Anno di riferimento", "anno"),
    "mese": ("Mese di riferimento", "1-12"),
    "mese_nome": ("Nome del mese per esteso", "testo"),
    "data": ("Data giornaliera del dato", "YYYY-MM-DD"),

    # --- Arrivi e presenze turistiche ---
    "arrivi": ("Numero di arrivi turistici grezzi (fonte originale)", "persone"),
    "arrivi_totali": ("Numero di arrivi turistici (persone che iniziano un soggiorno in struttura censita)", "persone/anno"),
    "presenze": ("Numero di presenze grezze (fonte originale)", "notti"),
    "presenze_totali": ("Numero di presenze (notti di pernottamento totali)", "notti/anno"),
    "presenze_mese": ("Presenze mensili reali per comune (dato SIRED, non stimato)", "notti/mese"),
    "presenze_anno_precedente": ("Presenze dell'anno precedente, per calcolo crescita YoY (colonna di appoggio)", "notti"),
    "macro-tipologia": ("Categoria struttura ricettiva (Alberghiero / Extra-alberghiero: Esercizi complementari / Extra-alberghiero: Alloggi privati in affitto)", "testo"),
    "provenienza": ("Paese o regione italiana di provenienza del turista", "testo"),

    # --- Capacità ricettiva ---
    "numero_strutture_totali": ("Numero totale di strutture ricettive attive", "strutture"),
    "numero_strutture": ("Numero di strutture (fonte originale, per tipologia/categoria)", "strutture"),
    "letti_totali": ("Numero totale di posti letto disponibili", "letti"),
    "letti": ("Numero posti letto (fonte originale, per tipologia/categoria)", "letti"),
    "camere_totali": ("Numero totale di camere disponibili", "camere"),
    "camere": ("Numero camere (fonte originale, per tipologia/categoria)", "camere"),
    "tipologia": ("Tipologia struttura (Albergo, Agriturismo, B&B, ecc.)", "testo"),
    "categoria": ("Categoria/classificazione struttura (es. stelle, quando applicabile)", "testo"),
    "letti_anno_precedente": ("Letti totali dell'anno precedente, per calcolo crescita YoY (colonna di appoggio)", "letti"),

    # --- Demografia e territorio ---
    "popolazione_residente": ("Popolazione residente ISTAT", "persone"),
    "superficie_kmq": ("Superficie comunale", "km²"),

    # --- Abitazioni (Censimento Permanente ISTAT) ---
    "abitazioni_occupate": ("Abitazioni occupate da residenti abituali", "abitazioni"),
    "abitazioni_non_occupate": ("Abitazioni non occupate da residenti (vuote o abitate solo part-time, es. seconde case) - Censimento 2021 per anno 2022, Censimento 2023 per 2023-2025", "abitazioni"),
    "abitazioni_totali": ("Totale abitazioni censite (occupate + non occupate)", "abitazioni"),

    # --- Porti e aeroporti ---
    "porto/aeroporto": ("Tipo di scalo: 'Porto' o 'Aeroporto'", "testo"),
    "nome": ("Nome dello scalo (es. Cagliari, Olbia, Alghero...)", "testo"),
    "partenze": ("Numero partenze giornaliere dallo scalo", "persone/giorno"),
    "arrivi_aeroporti": ("Arrivi giornalieri sommati sui 3 aeroporti sardi", "persone/giorno"),
    "arrivi_porti": ("Arrivi giornalieri sommati sui porti sardi", "persone/giorno"),
    "arrivi_totali_sardegna": ("Arrivi giornalieri totali Sardegna (porti+aeroporti sommati)", "persone/giorno"),
    "partenze_totali_sardegna": ("Partenze giornaliere totali Sardegna", "persone/giorno"),
    "arrivi_porti_aeroporti": ("Arrivi annuali totali via porto/aeroporto (flusso fisico di ingresso nell'isola)", "persone/anno"),
    "arrivi_sired": ("Arrivi annuali registrati in strutture ricettive censite (SIRED)", "persone/anno"),
    "flusso_non_catturato": ("Differenza arrivi_porti_aeroporti - arrivi_sired: flusso non spiegato da SIRED. ATTENZIONE: include insieme escursionisti, alloggi non censiti, residenti in transito, pendolari - non scorporabile", "persone/anno"),
    "quota_non_catturata_pct": ("Percentuale di flusso non catturato sul totale arrivi porti/aeroporti", "%"),

    # --- Stagionalità (Gini) ---
    "gini_zero_filled": ("Indice di Gini di stagionalità, mesi mancanti trattati come zero presenze", "indice 0-1"),
    "gini_solo_mesi_attivi": ("Indice di Gini calcolato solo sui mesi con dati reali (NULL se <3 mesi disponibili)", "indice 0-1"),
    "gini_stagionalita": ("Indice di Gini di stagionalità (versione zero-filled) nella tabella finale", "indice 0-1"),
    "gini_stagionalita_solo_mesi_attivi": ("Indice di Gini di stagionalità (versione solo mesi attivi) nella tabella finale", "indice 0-1"),
    "n_mesi_con_dati": ("Numero di mesi su 12 con almeno un dato di presenze reale", "1-12"),
    "n_mesi_con_dati_turismo": ("Come n_mesi_con_dati, nella tabella finale", "1-12"),

    # --- Indicatori derivati (presentation) ---
    "presenze_per_residente": ("Presenze turistiche annue per ogni residente - indicatore di pressione turistica", "notti/residente"),
    "presenze_per_kmq": ("Presenze turistiche annue per km² - intensità territoriale", "notti/km²"),
    "tasso_occupazione_media": ("Presenze reali / posti letto teorici disponibili nell'anno (diluito su 365 giorni)", "rapporto 0-1"),
    "permanenza_media_giorni": ("Notti medie di soggiorno per arrivo (presenze/arrivi)", "notti"),
    "crescita_presenze_yoy_pct": ("Variazione % presenze rispetto all'anno precedente (NULL per il primo anno 2022)", "%"),
    "crescita_letti_yoy_pct": ("Variazione % letti totali rispetto all'anno precedente (NULL per il primo anno 2022)", "%"),
    "letti_per_struttura": ("Dimensione media delle strutture ricettive", "letti/struttura"),
    "letti_per_residente": ("Capacità ricettiva ufficiale per residente", "letti/residente"),
    "quota_abitazioni_non_occupate_pct": ("Percentuale di abitazioni non occupate da residenti sul totale - proxy seconde case/affitti brevi", "%"),
    "posti_letto_informali_stimati": ("Stima posti letto informali: abitazioni_non_occupate × 2.3 persone/abitazione (coefficiente ISTAT dichiarato)", "posti stimati"),
    "pressione_potenziale_totale": ("(letti ufficiali + posti letto informali stimati) / popolazione residente", "posti/residente"),
    "densita_abitanti_kmq": ("Popolazione residente / superficie comunale", "abitanti/km²"),
}

# Costruzione del catalogo completo
df_catalogo = df_struttura.copy()
df_catalogo["descrizione"] = df_catalogo["column_name"].map(lambda c: descrizioni_colonne.get(c, ("*** DA COMPLETARE ***", "***"))[0])
df_catalogo["unita_misura"] = df_catalogo["column_name"].map(lambda c: descrizioni_colonne.get(c, ("*** DA COMPLETARE ***", "***"))[1])

# Colonne senza descrizione trovata (da controllare)
mancanti = df_catalogo[df_catalogo["descrizione"] == "*** DA COMPLETARE ***"]
print(f"Colonne senza descrizione trovata: {len(mancanti)}")
if len(mancanti) > 0:
    print(mancanti[["table_schema", "table_name", "column_name"]].to_string())

print(f"\nCatalogo completo: {len(df_catalogo)} righe")
df_catalogo.head(20)

Colonne senza descrizione trovata: 3
   table_schema               table_name       column_name
32          raw  abitazioni_non_occupate   somma_controllo
33          raw  abitazioni_non_occupate        differenza
38          raw        anagrafica_comuni  codice_provincia

Catalogo completo: 109 righe


,table_schema,table_name,column_name,data_type,ordinal_position,descrizione,unita_misura
0,presentation,indicatori_comune_anno,comune,VARCHAR,1,"Nome del comune, forma canonica (Title Case) d...",testo
1,presentation,indicatori_comune_anno,anno,INTEGER,2,Anno di riferimento,anno
2,presentation,indicatori_comune_anno,popolazione_residente,BIGINT,3,Popolazione residente ISTAT,persone
3,presentation,indicatori_comune_anno,superficie_kmq,DOUBLE,4,Superficie comunale,km²
4,presentation,indicatori_comune_anno,arrivi_totali,DOUBLE,5,Numero di arrivi turistici (persone che inizia...,persone/anno
5,presentation,indicatori_comune_anno,presenze_totali,HUGEINT,6,Numero di presenze (notti di pernottamento tot...,notti/anno
6,presentation,indicatori_comune_anno,numero_strutture_totali,HUGEINT,7,Numero totale di strutture ricettive attive,strutture
7,presentation,indicatori_comune_anno,letti_totali,HUGEINT,8,Numero totale di posti letto disponibili,letti
8,presentation,indicatori_comune_anno,camere_totali,HUGEINT,9,Numero totale di camere disponibili,camere
9,presentation,indicatori_comune_anno,abitazioni_occupate,BIGINT,10,Abitazioni occupate da residenti abituali,abitazioni


In [53]:
# ============================================================
# CATALOGO DATI - Blocco C: Completamento e salvataggio
# ============================================================

descrizioni_colonne["somma_controllo"] = ("Colonna di servizio per verifica coerenza (occupate+non_occupate) - CANDIDATA ALLA RIMOZIONE, non usare in analisi", "-")
descrizioni_colonne["differenza"] = ("Colonna di servizio per verifica coerenza (totale - somma_controllo, atteso 0) - CANDIDATA ALLA RIMOZIONE, non usare in analisi", "-")
descrizioni_colonne["codice_provincia"] = ("Codice numerico storico della provincia", "codice")

df_catalogo["descrizione"] = df_catalogo["column_name"].map(lambda c: descrizioni_colonne.get(c, ("*** DA COMPLETARE ***", "***"))[0])
df_catalogo["unita_misura"] = df_catalogo["column_name"].map(lambda c: descrizioni_colonne.get(c, ("*** DA COMPLETARE ***", "***"))[1])

# Aggiungiamo la fonte originale e la frequenza di aggiornamento, per tabella (non per colonna)
info_tabelle = {
    "anagrafica_comuni": ("ISTAT - Elenco comuni italiani", "statica, aggiornamento raro"),
    "porti_aeroporti": ("Regione Sardegna - sardegnamobilita.it", "giornaliera"),
    "arrivi_presenze_sired": ("Regione Sardegna - Osservatorio Turismo (SIRED/Ross1000)", "mensile"),
    "capacita_ricettiva": ("Regione Sardegna - Osservatorio Turismo (SIRED/Ross1000)", "annuale"),
    "abitazioni_non_occupate": ("ISTAT - Censimento Permanente Popolazione e Abitazioni", "2019, 2021, 2023 (censimento continuo)"),
    "popolazione_residente": ("ISTAT - Bilancio demografico comunale", "annuale"),
    "superficie_comunale": ("ISTAT - Statistiche geografiche sui comuni", "statica"),
    "comuni_riferimento": ("Derivata da raw.anagrafica_comuni", "statica"),
    "arrivi_presenze_annuale": ("Derivata da raw.arrivi_presenze_sired (aggregazione)", "annuale"),
    "capacita_annuale": ("Derivata da raw.capacita_ricettiva (aggregazione)", "annuale"),
    "abitazioni": ("Derivata da raw.abitazioni_non_occupate (pulizia + filtro Sardegna)", "2019/2021/2023"),
    "popolazione": ("Derivata da raw.popolazione_residente (pulizia + filtro Sardegna)", "annuale"),
    "superficie": ("Derivata da raw.superficie_comunale (pulizia + filtro Sardegna)", "statica"),
    "presenze_mensili": ("Derivata da raw.arrivi_presenze_sired (aggregazione mensile)", "mensile"),
    "gini_stagionalita": ("Calcolata in Python da staging.presenze_mensili", "annuale"),
    "flusso_regionale_giornaliero": ("Derivata da raw.porti_aeroporti (aggregazione)", "giornaliera"),
    "flusso_non_catturato_annuale": ("Calcolata da staging.arrivi_presenze_annuale + raw.porti_aeroporti", "annuale"),
    "indicatori_comune_anno": ("Tabella finale - join di tutte le staging + indicatori derivati", "annuale (con Gini su base mensile)"),
}

df_catalogo["fonte_originale"] = df_catalogo["table_name"].map(lambda t: info_tabelle.get(t, ("***", "***"))[0])
df_catalogo["frequenza_aggiornamento"] = df_catalogo["table_name"].map(lambda t: info_tabelle.get(t, ("***", "***"))[1])

# Riordino colonne per leggibilità
df_catalogo_finale = df_catalogo[[
    "table_schema", "table_name", "column_name", "data_type", 
    "descrizione", "unita_misura", "fonte_originale", "frequenza_aggiornamento"
]].rename(columns={
    "table_schema": "schema",
    "table_name": "tabella",
    "column_name": "colonna",
    "data_type": "tipo_dato"
})

# Salvataggio
PATH_CATALOGO = PRESENTATION_DIR / "catalogo_dati.csv"
df_catalogo_finale.to_csv(PATH_CATALOGO, index=False, encoding='utf-8')
print(f"Catalogo salvato in: {PATH_CATALOGO}")
print(f"Righe: {len(df_catalogo_finale)}")

# Verifica finale: nessuna colonna dovrebbe più avere descrizione mancante
controllo_finale = df_catalogo_finale[df_catalogo_finale["descrizione"] == "*** DA COMPLETARE ***"]
print(f"Colonne ancora senza descrizione: {len(controllo_finale)}")

df_catalogo_finale.head(15)

Catalogo salvato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\presentation\catalogo_dati.csv
Righe: 109
Colonne ancora senza descrizione: 0


,schema,tabella,colonna,tipo_dato,descrizione,unita_misura,fonte_originale,frequenza_aggiornamento
0,presentation,indicatori_comune_anno,comune,VARCHAR,"Nome del comune, forma canonica (Title Case) d...",testo,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
1,presentation,indicatori_comune_anno,anno,INTEGER,Anno di riferimento,anno,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
2,presentation,indicatori_comune_anno,popolazione_residente,BIGINT,Popolazione residente ISTAT,persone,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
3,presentation,indicatori_comune_anno,superficie_kmq,DOUBLE,Superficie comunale,km²,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
4,presentation,indicatori_comune_anno,arrivi_totali,DOUBLE,Numero di arrivi turistici (persone che inizia...,persone/anno,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
5,presentation,indicatori_comune_anno,presenze_totali,HUGEINT,Numero di presenze (notti di pernottamento tot...,notti/anno,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
6,presentation,indicatori_comune_anno,numero_strutture_totali,HUGEINT,Numero totale di strutture ricettive attive,strutture,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
7,presentation,indicatori_comune_anno,letti_totali,HUGEINT,Numero totale di posti letto disponibili,letti,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
8,presentation,indicatori_comune_anno,camere_totali,HUGEINT,Numero totale di camere disponibili,camere,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)
9,presentation,indicatori_comune_anno,abitazioni_occupate,BIGINT,Abitazioni occupate da residenti abituali,abitazioni,Tabella finale - join di tutte le staging + in...,annuale (con Gini su base mensile)


In [ ]:
# ============================================================
# FASE 2 - EXPORT: Tabella aggiornata con Gini per Tableau
# ============================================================

df_finale = con.execute("SELECT * FROM presentation.indicatori_comune_anno").df()

PATH_EXPORT_CSV = PRESENTATION_DIR / "indicatori_comune_anno.csv"
df_finale.to_csv(PATH_EXPORT_CSV, index=False, encoding='utf-8')
print(f"Esportato in: {PATH_EXPORT_CSV}")
print(f"Righe: {len(df_finale)}, Colonne: {len(df_finale.columns)}")

Esportato in: C:\Users\alecr\Documents\AIDA_2026_Data_Quality\Project Work AIDA26 - Gruppo 3\sardegna_overtourism\data\presentation\indicatori_comune_anno.csv
Righe: 1508, Colonne: 27


In [66]:
# ============================================================
# TABELLA COMPARATIVA - Cagliari, Villasimius, Sassari (2025)
# ============================================================
import pandas as pd

comuni_confronto = ["Cagliari", "Villasimius", "Sassari"]

df_confronto = con.execute(f"""
    SELECT *
    FROM presentation.indicatori_comune_anno
    WHERE comune IN ({','.join([f"'{c}'" for c in comuni_confronto])})
    AND anno = 2025
""").df()

# Trasponiamo: comuni come colonne, indicatori come righe, per leggerli affiancati
df_trasposta = df_confronto.set_index("comune").T
df_trasposta = df_trasposta[comuni_confronto]  # ordine fisso: Cagliari, Villasimius, Sassari

# Etichette leggibili per ogni riga (usando le descrizioni del catalogo dati)
etichette = {
    "popolazione_residente": "Popolazione residente",
    "superficie_kmq": "Superficie (km²)",
    "arrivi_totali": "Arrivi turistici",
    "presenze_totali": "Presenze (notti)",
    "numero_strutture_totali": "N. strutture ricettive",
    "letti_totali": "Posti letto",
    "camere_totali": "Camere",
    "abitazioni_non_occupate": "Abitazioni non occupate",
    "abitazioni_totali": "Abitazioni totali",
    "presenze_per_residente": "Presenze per residente",
    "presenze_per_kmq": "Presenze per km²",
    "tasso_occupazione_media": "Tasso occupazione media",
    "permanenza_media_giorni": "Permanenza media (notti)",
    "crescita_presenze_yoy_pct": "Crescita presenze YoY (%)",
    "crescita_letti_yoy_pct": "Crescita letti YoY (%)",
    "letti_per_struttura": "Letti per struttura",
    "letti_per_residente": "Letti per residente",
    "quota_abitazioni_non_occupate_pct": "Quota abitazioni non occupate (%)",
    "posti_letto_informali_stimati": "Posti letto informali (stima)",
    "pressione_potenziale_totale": "Pressione potenziale totale",
    "densita_abitanti_kmq": "Densità abitanti/km²",
    "gini_stagionalita": "Indice Gini stagionalità",
    "n_mesi_con_dati_turismo": "Mesi con dati turismo",
}

righe_da_mostrare = [k for k in etichette.keys() if k in df_trasposta.index]
df_finale = df_trasposta.loc[righe_da_mostrare].copy()
df_finale.index = [etichette[k] for k in righe_da_mostrare]

# Formattazione numerica leggibile per riga
def formatta_riga(row):
    nome = row.name
    if "%" in nome or "Tasso" in nome:
        return row.apply(lambda v: f"{v:.1f}" if pd.notna(v) else "—")
    elif "Indice" in nome or "Permanenza" in nome or "per residente" in nome.lower() or "per km" in nome.lower() or "Letti per struttura" in nome:
        return row.apply(lambda v: f"{v:,.2f}" if pd.notna(v) else "—")
    else:
        return row.apply(lambda v: f"{v:,.0f}" if pd.notna(v) else "—")

df_visualizzata = df_finale.apply(formatta_riga, axis=1)

# Styling: intestazioni scure, righe alternate, testo allineato a destra per i numeri
styled = (
    df_visualizzata.style
    .set_properties(**{"text-align": "right", "padding": "6px 12px"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#1B4B5A"), ("color", "white"), 
                                       ("text-align", "center"), ("padding", "8px 12px")]},
        {"selector": "th.row_heading", "props": [("text-align", "left"), ("background-color", "#f0f0f0"),
                                                    ("color", "#333333")]},
        {"selector": "tr:nth-child(even)", "props": [("background-color", "#130D0D")]},
    ])
    .set_caption("Confronto indicatori 2025 — Cagliari, Villasimius, Sassari")
)

styled

comune,Cagliari,Villasimius,Sassari
Popolazione residente,"146,692","3,735","120,510"
Superficie (km²),85,58,547
Arrivi turistici,"442,982","172,783","107,337"
Presenze (notti),"1,123,435","916,156","274,114"
N. strutture ricettive,"2,952","1,793",723
Posti letto,"17,215","17,027","4,725"
Camere,"8,138","6,765","2,166"
Abitazioni non occupate,"7,286","4,541","8,068"
Abitazioni totali,"82,245","6,482","65,535"
Presenze per residente,7.66,245.29,2.27


In [67]:
# ============================================================
# TOP COMUNI - Tasso occupazione media e Pressione potenziale totale (2025)
# ============================================================

# --- Top 15 per Tasso di Occupazione Media ---
df_top_occupazione = con.execute("""
    SELECT 
        comune, 
        tasso_occupazione_media,
        presenze_totali,
        letti_totali,
        popolazione_residente
    FROM presentation.indicatori_comune_anno
    WHERE anno = 2025 AND tasso_occupazione_media IS NOT NULL
    ORDER BY tasso_occupazione_media DESC
    LIMIT 15
""").df()

print("TOP 15 - Tasso di occupazione media (2025):")
print(df_top_occupazione.to_string(index=False))

print("\n" + "="*80 + "\n")

# --- Top 15 per Pressione Potenziale Totale ---
df_top_pressione_potenziale = con.execute("""
    SELECT 
        comune, 
        pressione_potenziale_totale,
        letti_totali,
        posti_letto_informali_stimati,
        popolazione_residente,
        quota_abitazioni_non_occupate_pct
    FROM presentation.indicatori_comune_anno
    WHERE anno = 2025 AND pressione_potenziale_totale IS NOT NULL
    ORDER BY pressione_potenziale_totale DESC
    LIMIT 15
""").df()

print("TOP 15 - Pressione potenziale totale (2025):")
print(df_top_pressione_potenziale.to_string(index=False))

TOP 15 - Tasso di occupazione media (2025):
      comune  tasso_occupazione_media  presenze_totali  letti_totali  popolazione_residente
         Uta                    1.845         129268.0         192.0                   8933
Fordongianus                    0.525          42531.0         222.0                    812
     Sardara                    0.353          52987.0         411.0                   3744
    Tramatza                    0.297           5752.0          53.0                    897
     Macomer                    0.288          12407.0         118.0                   9069
       Turri                    0.287           2198.0          21.0                    372
     Tortolì                    0.257         890584.0        9512.0                  10978
       Nurri                    0.251           3109.0          34.0                   2021
     Cardedu                    0.247         192105.0        2134.0                   1983
      Ottana                    0.24

In [68]:
# ============================================================
# DISAGGREGAZIONE - Blocco 1: Validazione forma mensile
# ============================================================
import numpy as np

# Forma mensile regionale (somma tutti gli scali, quota % di ogni mese sul totale annuo)
df_regionale_mensile = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM data) AS anno,
        EXTRACT(MONTH FROM data) AS mese,
        SUM(arrivi) AS arrivi_mese
    FROM raw.porti_aeroporti
    WHERE EXTRACT(YEAR FROM data) IN (2022,2023,2024,2025)
    GROUP BY EXTRACT(YEAR FROM data), EXTRACT(MONTH FROM data)
""").df()

def normalizza_quote(df, col_valore, col_gruppo="anno"):
    df = df.copy()
    df["quota"] = df.groupby(col_gruppo)[col_valore].transform(lambda x: x / x.sum())
    return df

df_regionale_mensile = normalizza_quote(df_regionale_mensile, "arrivi_mese")

# Comuni campione: alta stagionalità nota, bassa stagionalità nota, e uno intermedio
comuni_campione = ["Villasimius", "Sassari", "Cagliari", "Baunei", "Olbia"]

df_mensile_comuni = con.execute(f"""
    SELECT comune, anno, mese, presenze_mese
    FROM staging.presenze_mensili
    WHERE comune IN ({','.join([f"'{c}'" for c in comuni_campione])})
    AND anno IN (2022,2023,2024,2025)
""").df()
df_mensile_comuni = normalizza_quote(df_mensile_comuni, "presenze_mese", col_gruppo=["comune","anno"])

# Correlazione tra forma regionale e forma di ciascun comune, anno per anno
risultati_correlazione = []
for comune in comuni_campione:
    for anno in [2022,2023,2024,2025]:
        serie_comune = df_mensile_comuni[(df_mensile_comuni["comune"]==comune) & (df_mensile_comuni["anno"]==anno)].set_index("mese")["quota"].reindex(range(1,13))
        serie_regionale = df_regionale_mensile[df_regionale_mensile["anno"]==anno].set_index("mese")["quota"].reindex(range(1,13))
        if serie_comune.notna().sum() >= 8:  # richiediamo almeno 8 mesi validi per una correlazione sensata
            corr = serie_comune.corr(serie_regionale)
            risultati_correlazione.append({"comune": comune, "anno": anno, "correlazione_con_regionale": round(corr,3)})

df_correlazioni = pd.DataFrame(risultati_correlazione)
print(df_correlazioni.pivot(index="comune", columns="anno", values="correlazione_con_regionale"))

anno          2022   2023   2024   2025
comune                                 
Baunei       0.979  0.984  0.961  0.982
Cagliari     0.954  0.948  0.933  0.958
Olbia        0.991  0.992  0.977  0.992
Sassari      0.849  0.880  0.816  0.926
Villasimius  0.990  0.984  0.976  0.982


In [70]:
# ============================================================
# DISAGGREGAZIONE - Blocco 2: Segmentazione su TUTTI i comuni
# ============================================================

# Calcoliamo la correlazione con la curva regionale per TUTTI i comuni con almeno 8 mesi di dati,
# usando la media delle quote sui 4 anni disponibili (più stabile del singolo anno)

df_tutti_mensili = con.execute("""
    SELECT comune, anno, mese, presenze_mese
    FROM staging.presenze_mensili
""").df()
df_tutti_mensili = normalizza_quote(df_tutti_mensili, "presenze_mese", col_gruppo=["comune","anno"])

quota_regionale_media = df_regionale_mensile.groupby("mese")["quota"].mean()

risultati_tutti = []
for comune, gruppo in df_tutti_mensili.groupby("comune"):
    quota_comune_media = gruppo.groupby("mese")["quota"].mean().reindex(range(1,13))
    n_mesi_validi = quota_comune_media.notna().sum()
    if n_mesi_validi >= 8:
        corr = quota_comune_media.corr(quota_regionale_media.reindex(range(1,13)))
        risultati_tutti.append({"comune": comune, "correlazione_regionale": round(corr,3), "n_mesi_validi": n_mesi_validi})

df_segmentazione = pd.DataFrame(risultati_tutti)

# Soglia empirica basata sul campione appena visto: sopra 0.90 = "allineato al flusso turistico regionale"
df_segmentazione["profilo"] = df_segmentazione["correlazione_regionale"].apply(
    lambda c: "Allineato al flusso turistico (costiero/balneare)" if c >= 0.90 
    else "Parzialmente allineato (misto)" if c >= 0.75 
    else "Non allineato (urbano/atipico)"
)

print(df_segmentazione["profilo"].value_counts())
print(f"\nEsempi per categoria:")
for profilo in df_segmentazione["profilo"].unique():
    esempi = df_segmentazione[df_segmentazione["profilo"]==profilo].nlargest(5, "correlazione_regionale")["comune"].tolist()
    print(f"  {profilo}: {esempi}")

profilo
Allineato al flusso turistico (costiero/balneare)    117
Non allineato (urbano/atipico)                        70
Parzialmente allineato (misto)                        63
Name: count, dtype: int64

Esempi per categoria:
  Non allineato (urbano/atipico): ['Baressa', 'Pimentel', 'Gonnostramatza', 'Orani', 'Bottidda']
  Parzialmente allineato (misto): ['Luras', 'Bitti', 'Decimomannu', 'Martis', 'Lunamatrona']
  Allineato al flusso turistico (costiero/balneare): ['Olbia', 'Cardedu', 'Cuglieri', 'Magomadas', 'Palau']


In [71]:
# ============================================================
# DISAGGREGAZIONE - Blocco 3: Mappatura geografica e curve per gruppo
# ============================================================

# Mappatura provincia -> scali di riferimento (approssimazione geografica dichiarata)
MAPPA_PROVINCIA_SCALI = {
    "SS": ["Alghero", "Porto Torres"],           # Sassari: nord-ovest
    "NU": ["Olbia", "Arbatax"],                   # Nuoro: costa orientale
    "CA": ["Cagliari"],                           # Cagliari
    "SU": ["Cagliari"],                           # Sud Sardegna
    "OR": ["Cagliari", "Alghero"],                 # Oristano: centrale, hub più vicini
}

# Curva giornaliera per singolo scalo (quota % sul totale mensile dello scalo stesso)
df_scali_giornaliero = con.execute("""
    SELECT data, nome AS scalo, SUM(arrivi) AS arrivi_giorno
    FROM raw.porti_aeroporti
    GROUP BY data, nome
""").df()
df_scali_giornaliero["anno"] = pd.to_datetime(df_scali_giornaliero["data"]).dt.year
df_scali_giornaliero["mese"] = pd.to_datetime(df_scali_giornaliero["data"]).dt.month
df_scali_giornaliero["giorno"] = pd.to_datetime(df_scali_giornaliero["data"]).dt.day

df_scali_giornaliero["quota_giorno_nel_mese"] = df_scali_giornaliero.groupby(
    ["scalo", "anno", "mese"]
)["arrivi_giorno"].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)

# Curva regionale aggregata (già calcolata concettualmente, la ricostruiamo a livello giornaliero)
df_regionale_giornaliero = con.execute("""
    SELECT data, SUM(arrivi) AS arrivi_giorno
    FROM raw.porti_aeroporti
    GROUP BY data
""").df()
df_regionale_giornaliero["anno"] = pd.to_datetime(df_regionale_giornaliero["data"]).dt.year
df_regionale_giornaliero["mese"] = pd.to_datetime(df_regionale_giornaliero["data"]).dt.month
df_regionale_giornaliero["quota_giorno_nel_mese"] = df_regionale_giornaliero.groupby(
    ["anno", "mese"]
)["arrivi_giorno"].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)

print("Curve giornaliere pronte:")
print(f"  Per scalo: {df_scali_giornaliero['scalo'].nunique()} scali, {len(df_scali_giornaliero)} righe")
print(f"  Regionale: {len(df_regionale_giornaliero)} righe")

# Assegnazione comune -> metodo e curva di riferimento
df_comuni_provincia = con.execute("SELECT comune, provincia_sigla FROM staging.comuni_riferimento").df()
df_assegnazione = df_comuni_provincia.merge(df_segmentazione, on="comune", how="left")
df_assegnazione["profilo"] = df_assegnazione["profilo"].fillna("Dati insufficienti")

def assegna_metodo(row):
    if row["profilo"] == "Allineato al flusso turistico (costiero/balneare)":
        return "curva_scalo"
    elif row["profilo"] == "Parzialmente allineato (misto)":
        return "curva_regionale"
    else:
        return "uniforme"

df_assegnazione["metodo_disaggregazione"] = df_assegnazione.apply(assegna_metodo, axis=1)
df_assegnazione["scali_riferimento"] = df_assegnazione["provincia_sigla"].map(MAPPA_PROVINCIA_SCALI)

print(f"\nDistribuzione metodi:")
print(df_assegnazione["metodo_disaggregazione"].value_counts())
df_assegnazione.head(10)

Curve giornaliere pronte:
  Per scalo: 7 scali, 8953 righe
  Regionale: 1651 righe

Distribuzione metodi:
metodo_disaggregazione
uniforme           197
curva_scalo        117
curva_regionale     63
Name: count, dtype: int64


,comune,provincia_sigla,correlazione_regionale,n_mesi_validi,profilo,metodo_disaggregazione,scali_riferimento
0,Aggius,SS,0.873,10.0,Parzialmente allineato (misto),curva_regionale,"[Alghero, Porto Torres]"
1,Alà dei Sardi,SS,NaN,NaN,Dati insufficienti,uniforme,"[Alghero, Porto Torres]"
2,Alghero,SS,0.981,12.0,Allineato al flusso turistico (costiero/balneare),curva_scalo,"[Alghero, Porto Torres]"
3,Anela,SS,NaN,NaN,Dati insufficienti,uniforme,"[Alghero, Porto Torres]"
4,Ardara,SS,NaN,NaN,Dati insufficienti,uniforme,"[Alghero, Porto Torres]"
5,Arzachena,SS,0.979,12.0,Allineato al flusso turistico (costiero/balneare),curva_scalo,"[Alghero, Porto Torres]"
6,Banari,SS,0.789,10.0,Parzialmente allineato (misto),curva_regionale,"[Alghero, Porto Torres]"
7,Benetutti,SS,0.821,12.0,Parzialmente allineato (misto),curva_regionale,"[Alghero, Porto Torres]"
8,Berchidda,SS,0.965,12.0,Allineato al flusso turistico (costiero/balneare),curva_scalo,"[Alghero, Porto Torres]"
9,Bessude,SS,NaN,NaN,Dati insufficienti,uniforme,"[Alghero, Porto Torres]"


In [76]:
# ============================================================
# DISAGGREGAZIONE - Blocco 4: Coordinate geografiche
# ============================================================
import pandas as pd
import numpy as np

# Coordinate dei 9 scali (aeroporti verificati via fonti ufficiali/Wikipedia;
# porti coincidenti con un comune usano lo stesso centroide del comune per coerenza;
# Arbatax e Portovesme, non essendo comuni a sé, usano coordinate verificate via ricerca web)
COORDINATE_SCALI = {
    "Aeroporto_Cagliari":      (39.2515, 9.0543),
    "Aeroporto_Olbia":         (40.8858, 9.5169),
    "Aeroporto_Alghero":       (40.6311, 8.2886),
    "Porto_Cagliari":          (39.2247, 9.0885),
    "Porto_Olbia":             (40.9232, 9.4803),
    "Porto_Porto Torres":      (40.9398, 8.3216),
    "Porto_Golfo Aranci":      (40.9873, 9.5740),
    "Porto_Arbatax":           (39.9402, 9.7011),
    "Porto_Porto Vesme":       (39.1897, 8.3714),
}

df_scali_coord = pd.DataFrame([
    {"scalo": k, "lat": v[0], "lon": v[1]} for k, v in COORDINATE_SCALI.items()
])
print("Coordinate scali:")
print(df_scali_coord)

# Coordinate dei 377 comuni (calcolate come centroide del confine amministrativo,
# fonte: OpenPolis/ISTAT geojson dei confini comunali)
df_comuni_coord = pd.read_csv(RAW_DIR / "originali" / "centroidi_sardegna.csv")
print(f"Comuni con coordinate: {len(df_comuni_coord)}")
df_comuni_coord.head()

Coordinate scali:
                scalo      lat     lon
0  Aeroporto_Cagliari  39.2515  9.0543
1     Aeroporto_Olbia  40.8858  9.5169
2   Aeroporto_Alghero  40.6311  8.2886
3      Porto_Cagliari  39.2247  9.0885
4         Porto_Olbia  40.9232  9.4803
5  Porto_Porto Torres  40.9398  8.3216
6  Porto_Golfo Aranci  40.9873  9.5740
7       Porto_Arbatax  39.9402  9.7011
8   Porto_Porto Vesme  39.1897  8.3714
Comuni con coordinate: 377


,codice_istat,nome,lat,lon
0,90001,Aggius,40.967551,9.027880
1,90002,Alà dei Sardi,40.678845,9.342916
2,90003,Alghero,40.600273,8.298833
3,90004,Anela,40.452679,9.032644
4,90005,Ardara,40.628508,8.821478


In [77]:
# ============================================================
# DISAGGREGAZIONE - Blocco 5: Distanza e correlazione comune-scalo
# ============================================================

def distanza_km(lat1, lon1, lat2, lon2):
    """Formula di Haversine - distanza in linea d'aria in km tra due punti geografici"""
    R = 6371  # raggio terrestre in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Serie mensile per singolo scalo (quota % sul totale annuo dello scalo, media sui 4 anni)
df_scalo_mensile = con.execute("""
    SELECT 
        CASE WHEN "porto/aeroporto"='Aeroporto' THEN 'Aeroporto_' || nome ELSE 'Porto_' || nome END AS scalo,
        EXTRACT(YEAR FROM data) AS anno,
        EXTRACT(MONTH FROM data) AS mese,
        SUM(arrivi) AS arrivi_mese
    FROM raw.porti_aeroporti
    GROUP BY scalo, EXTRACT(YEAR FROM data), EXTRACT(MONTH FROM data)
""").df()
df_scalo_mensile["quota"] = df_scalo_mensile.groupby(["scalo","anno"])["arrivi_mese"].transform(lambda x: x/x.sum())
quota_media_scalo = df_scalo_mensile.groupby(["scalo","mese"])["quota"].mean().reset_index()

# Serie mensile media per comune (già calcolata concettualmente prima, la ricostruiamo pulita)
quota_media_comune = df_tutti_mensili.groupby(["comune","mese"])["quota"].mean().reset_index()

# Calcolo distanza + correlazione per OGNI coppia comune-scalo
righe_modello = []
for _, riga_comune in df_comuni_coord.iterrows():
    comune = riga_comune["nome"]
    serie_c = quota_media_comune[quota_media_comune["comune"]==comune].set_index("mese")["quota"].reindex(range(1,13))
    n_mesi_validi = serie_c.notna().sum()
    
    for _, riga_scalo in df_scali_coord.iterrows():
        scalo = riga_scalo["scalo"]
        dist = distanza_km(riga_comune["lat"], riga_comune["lon"], riga_scalo["lat"], riga_scalo["lon"])
        
        serie_s = quota_media_scalo[quota_media_scalo["scalo"]==scalo].set_index("mese")["quota"].reindex(range(1,13))
        corr = serie_c.corr(serie_s) if n_mesi_validi >= 6 else np.nan
        
        righe_modello.append({
            "comune": comune, "scalo": scalo, 
            "distanza_km": round(dist,1), 
            "correlazione": round(corr,3) if pd.notna(corr) else None,
            "n_mesi_validi": n_mesi_validi
        })

df_modello = pd.DataFrame(righe_modello)
print(f"Coppie comune-scalo calcolate: {len(df_modello)} ({df_comuni_coord.shape[0]} comuni x {df_scali_coord.shape[0]} scali)")
df_modello[df_modello["comune"]=="Villasimius"].sort_values("distanza_km")

C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stdd

Coppie comune-scalo calcolate: 3393 (377 comuni x 9 scali)


C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\alecr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stdd

,comune,scalo,distanza_km,correlazione,n_mesi_validi
3369,Villasimius,Porto_Cagliari,37.2,0.911,12
3366,Villasimius,Aeroporto_Cagliari,40.7,0.841,12
3373,Villasimius,Porto_Arbatax,88.8,0.843,12
3374,Villasimius,Porto_Porto Vesme,98.3,-0.251,12
3367,Villasimius,Aeroporto_Olbia,192.4,0.962,12
3368,Villasimius,Aeroporto_Alghero,194.4,0.860,12
3370,Villasimius,Porto_Olbia,196.6,0.957,12
3372,Villasimius,Porto_Golfo Aranci,203.8,0.944,12
3371,Villasimius,Porto_Porto Torres,222.7,0.959,12


In [102]:
# ============================================================
# DISAGGREGAZIONE - Blocco 6: Pesi probabilistici comune-scalo
# ============================================================

DECAY_KM = 80  # distanza (km) alla quale il peso geografico si dimezza circa - parametro dichiarato, modificabile

def calcola_pesi(df_modello, decay_km=DECAY_KM):
    df = df_modello.copy()
    
    # Componente distanza: decadimento esponenziale (vicino = peso alto, lontano = peso ~0)
    df["peso_distanza"] = np.exp(-df["distanza_km"] / decay_km)
    
    # Componente correlazione: usata come moltiplicatore, azzerando le correlazioni negative
    # (scali comportamentalmente incoerenti col comune, es. Porto Vesme industriale)
    df["peso_correlazione"] = df["correlazione"].apply(lambda c: max(c, 0) if pd.notna(c) else 0.5)  # 0.5 = neutro se dati insufficienti
    
    # Punteggio combinato
    df["punteggio"] = df["peso_distanza"] * df["peso_correlazione"]
    
    # Normalizzazione: per ogni comune, i punteggi sui 9 scali sommano a 1 (probabilità)
    df["probabilita"] = df.groupby("comune")["punteggio"].transform(lambda x: x / x.sum() if x.sum() > 0 else 0)
    
    return df

df_pesi = calcola_pesi(df_modello)

comune_test= "Ottana"
print(f"Distribuzione probabilità sugli scali per {comune_test}:")
print(df_pesi[df_pesi["comune"]==comune_test].sort_values("probabilita", ascending=False)[
    ["scalo","distanza_km","correlazione","probabilita"]
].to_string(index=False))

Distribuzione probabilità sugli scali per Ottana:
             scalo  distanza_km  correlazione  probabilita
 Porto_Porto Vesme        128.6         0.508          1.0
Aeroporto_Cagliari        108.4        -0.351          0.0
   Aeroporto_Olbia         83.7        -0.373          0.0
    Porto_Cagliari        111.5        -0.337          0.0
 Aeroporto_Alghero         77.8        -0.364          0.0
       Porto_Olbia         86.0        -0.398          0.0
Porto_Porto Torres         99.8        -0.375          0.0
Porto_Golfo Aranci         95.9        -0.422          0.0
     Porto_Arbatax         64.7        -0.403          0.0


In [94]:
# ============================================================
# DISAGGREGAZIONE - Blocco 7: Curva giornaliera pesata per comune
# ============================================================

# Curva giornaliera per singolo scalo (quota % del giorno sul totale mensile di quello scalo)
df_scalo_giornaliero = con.execute("""
    SELECT 
        CASE WHEN "porto/aeroporto"='Aeroporto' THEN 'Aeroporto_' || nome ELSE 'Porto_' || nome END AS scalo,
        data,
        SUM(arrivi) AS arrivi_giorno
    FROM raw.porti_aeroporti
    GROUP BY scalo, data
""").df()
df_scalo_giornaliero["data"] = pd.to_datetime(df_scalo_giornaliero["data"])
df_scalo_giornaliero["anno"] = df_scalo_giornaliero["data"].dt.year
df_scalo_giornaliero["mese"] = df_scalo_giornaliero["data"].dt.month
df_scalo_giornaliero["quota_giorno_nel_mese"] = df_scalo_giornaliero.groupby(
    ["scalo","anno","mese"]
)["arrivi_giorno"].transform(lambda x: x/x.sum() if x.sum()>0 else 0)

# Per ogni comune, calcoliamo la sua curva giornaliera come MEDIA PESATA delle curve dei 9 scali,
# usando le probabilità del Blocco 6 come pesi
def costruisci_curva_comune(comune, df_pesi, df_scalo_giornaliero):
    pesi_comune = df_pesi[df_pesi["comune"]==comune].set_index("scalo")["probabilita"]
    
    df_join = df_scalo_giornaliero.copy()
    df_join["peso"] = df_join["scalo"].map(pesi_comune)
    df_join["contributo"] = df_join["quota_giorno_nel_mese"] * df_join["peso"]
    
    curva = df_join.groupby(["data","anno","mese"])["contributo"].sum().reset_index()
    curva = curva.rename(columns={"contributo": "peso_giorno_nel_mese"})
    curva["comune"] = comune
    return curva

# Test su Villasimius
curva_villasimius = costruisci_curva_comune("Villasimius", df_pesi, df_scalo_giornaliero)
print("Curva Villasimius, agosto 2025 (dovrebbe sommare a 1 sul mese):")
agosto_2025 = curva_villasimius[(curva_villasimius["anno"]==2025) & (curva_villasimius["mese"]==8)]
print(agosto_2025[["data","peso_giorno_nel_mese"]].to_string(index=False))
print(f"\nSomma pesi agosto 2025: {agosto_2025['peso_giorno_nel_mese'].sum():.4f} (atteso: 1.0000)")

Curva Villasimius, agosto 2025 (dovrebbe sommare a 1 sul mese):
      data  peso_giorno_nel_mese
2025-08-01              0.035420
2025-08-02              0.039206
2025-08-03              0.052847
2025-08-04              0.014833
2025-08-05              0.037268
2025-08-06              0.034489
2025-08-07              0.033945
2025-08-08              0.035519
2025-08-09              0.044154
2025-08-10              0.057995
2025-08-11              0.014490
2025-08-12              0.042364
2025-08-13              0.033658
2025-08-14              0.034530
2025-08-15              0.031918
2025-08-16              0.036940
2025-08-17              0.058177
2025-08-18              0.013966
2025-08-19              0.035689
2025-08-20              0.025305
2025-08-21              0.023958
2025-08-22              0.024139
2025-08-23              0.029139
2025-08-24              0.046921
2025-08-25              0.012112
2025-08-26              0.024703
2025-08-27              0.018819
2025-08-28  

In [95]:
# ============================================================
# DETTAGLIO - Tabella completa scalo-per-comune con probabilità
# ============================================================

# Per ogni comune, prendiamo lo scalo con probabilità più alta, i primi 3, 
# e la probabilità cumulata dei primi 2 (quanto "concentrata" è l'assegnazione)
def riepilogo_comune(gruppo):
    ordinato = gruppo.sort_values("probabilita", ascending=False)
    top1 = ordinato.iloc[0]
    top2 = ordinato.iloc[1] if len(ordinato) > 1 else None
    top3 = ordinato.iloc[2] if len(ordinato) > 2 else None
    
    return pd.Series({
        "scalo_principale": top1["scalo"],
        "probabilita_principale": round(top1["probabilita"], 3),
        "scalo_secondario": top2["scalo"] if top2 is not None else None,
        "probabilita_secondario": round(top2["probabilita"], 3) if top2 is not None else None,
        "scalo_terziario": top3["scalo"] if top3 is not None else None,
        "probabilita_terziario": round(top3["probabilita"], 3) if top3 is not None else None,
        "concentrazione_top2_pct": round((top1["probabilita"] + (top2["probabilita"] if top2 is not None else 0)) * 100, 1),
        "distanza_scalo_principale_km": top1["distanza_km"],
    })

df_riepilogo_assegnazione = df_pesi.groupby("comune").apply(riepilogo_comune).reset_index()

print(f"Riepilogo per {len(df_riepilogo_assegnazione)} comuni")
print("\nEsempio - primi 15 comuni in ordine alfabetico:")
print(df_riepilogo_assegnazione.head(15).to_string(index=False))

# Salvataggio: sia il dettaglio completo (377 comuni x 9 scali = 3393 righe)
# sia il riepilogo sintetico (377 righe, uno scalo principale per comune)
PATH_DETTAGLIO_COMPLETO = PRESENTATION_DIR / "modello_assegnazione_scali_dettaglio_completo.csv"
PATH_RIEPILOGO = PRESENTATION_DIR / "modello_assegnazione_scali_riepilogo.csv"

df_pesi.to_csv(PATH_DETTAGLIO_COMPLETO, index=False, encoding='utf-8')
df_riepilogo_assegnazione.to_csv(PATH_RIEPILOGO, index=False, encoding='utf-8')

print(f"\nSalvati:")
print(f"  Dettaglio completo (tutte le coppie comune-scalo): {PATH_DETTAGLIO_COMPLETO}")
print(f"  Riepilogo sintetico (un scalo principale per comune): {PATH_RIEPILOGO}")

Riepilogo per 377 comuni

Esempio - primi 15 comuni in ordine alfabetico:
       comune   scalo_principale  probabilita_principale   scalo_secondario  probabilita_secondario    scalo_terziario  probabilita_terziario  concentrazione_top2_pct  distanza_scalo_principale_km
    Abbasanta  Aeroporto_Alghero                   0.234      Porto_Arbatax                   0.139 Porto_Porto Torres                  0.121                     37.3                          68.1
       Aggius        Porto_Olbia                   0.247 Porto_Golfo Aranci                   0.218    Aeroporto_Olbia                  0.208                     46.5                          38.3
     Aglientu        Porto_Olbia                   0.262 Porto_Golfo Aranci                   0.241    Aeroporto_Olbia                  0.228                     50.4                          36.7
 Aidomaggiore      Porto_Arbatax                   0.194  Aeroporto_Alghero                   0.179 Porto_Porto Torres                  0.

In [97]:
# ============================================================
# ANALISI - Distribuzione della concentrazione su tutti i 377 comuni
# ============================================================

print("Statistiche su concentrazione_top2_pct (tutti i 377 comuni):")
print(df_riepilogo_assegnazione["concentrazione_top2_pct"].describe())

print("\nQuanti comuni per fascia di affidabilità:")
bins = [0, 40, 55, 70, 100]
labels = ["Bassa (<40%)", "Media (40-55%)", "Buona (55-70%)", "Alta (>70%)"]
df_riepilogo_assegnazione["fascia_affidabilita"] = pd.cut(
    df_riepilogo_assegnazione["concentrazione_top2_pct"], bins=bins, labels=labels
)
print(df_riepilogo_assegnazione["fascia_affidabilita"].value_counts().sort_index())

print("\nComuni con affidabilità più alta (top 20):")
print(df_riepilogo_assegnazione.nlargest(20, "concentrazione_top2_pct")[["comune","scalo_principale","concentrazione_top2_pct"]].to_string(index=False))


Statistiche su concentrazione_top2_pct (tutti i 377 comuni):
count    377.000000
mean      54.427586
std       16.452935
min        0.000000
25%       41.700000
50%       51.500000
75%       63.200000
max      100.000000
Name: concentrazione_top2_pct, dtype: float64

Quanti comuni per fascia di affidabilità:
fascia_affidabilita
Bassa (<40%)       80
Media (40-55%)    132
Buona (55-70%)     96
Alta (>70%)        68
Name: count, dtype: int64

Comuni con affidabilità più alta (top 20):
          comune   scalo_principale  concentrazione_top2_pct
       Bortigali  Porto_Porto Vesme                    100.0
    Fordongianus  Porto_Porto Vesme                    100.0
          Genoni  Porto_Porto Vesme                    100.0
          Laconi  Porto_Porto Vesme                    100.0
          Ottana  Porto_Porto Vesme                    100.0
         Samassi    Aeroporto_Olbia                    100.0
       Nuraminis Aeroporto_Cagliari                     92.9
            Sini  Porto_